# CARA v6.2 — Hypothesis 2: Concept–token attention region attack

**Run top to bottom.** `Config` is the first code cell. Default pilot: 5 pairs, 300 updates, Grad-ECLIP. 200-pair run: edit `NUM_SAMPLES=200` after pilot passes.

**Research provenance:** Author functions are embedded unchanged, with SHA-256 verification, in the **Preprocessing / Author Code** cells. CARA's multi-negative InfoNCE, replacement-to-sentence bridge, model fusion, per-concept attention extraction and dynamic crop are **new integration code**, not presented as original paper code. Original repos: [FINER](https://github.com/ExplainableML/finer), [MPCAttack](https://github.com/LiYuanBoJNU/MPCAttack), [SAGA](https://github.com/jaehyun-kwak/SAGA), [M-Attack-V2](https://github.com/VILA-Lab/M-Attack-V2), [Grad-ECLIP](https://github.com/CassieWeng/Grad-ECLIP). The referenced implementation patterns are not claimed to reproduce entire baseline methods.

**Important:** source/target pairing must be checked in Data preview. Contrastive objectives / CLIP cosine are proxies; targeted attack success requires independent caption evaluation and human labels. CUDA end-to-end validation must be carried out in Kaggle.

## Config

In [ ]:
# Configuration — edit this cell, then Run all.
from pathlib import Path
INSTALL_DEPENDENCIES = True
RUN_EXPERIMENT = True
RUN_MODE = "pilot"                 # artifact label: "pilot" | "full"; count uses NUM_SAMPLES
NUM_SAMPLES = 2                     # pilot5; optionally1 for smoke,200 or1000 for full
XAI_METHOD = "grad_eclip"           # "grad_eclip" | "qwen_attention"
DATASET_ROOT = Path('/kaggle/input/blackbox-adversarial-attack-dataset')  # attached Kaggle dataset
SOURCE_DIR = None                    # auto-discover exactly one "bigscale" directory
TARGET_DIR = None                    # auto-discover exactly one "target_images" directory
PAIR_MANIFEST = None                 # JSONL: {"index": 0, "source": "...", "target": "..."}
OUTPUT_ROOT = Path('/kaggle/working/kltn_attack_v6_cara')
SAMPLE_START = 0
SEED = 2026
K = 5
STEPS = 300
REFRESH_EVERY = 30
RANDOM_FROM_STEP = 150               # 1-based: updates 150..300 random, XAI still measured
EPSILON_255 = 16                     # CARA default, not specified by hypothesis
ALPHA_255 = 1                        # MPC pixel-space momentum ascent
TEMPERATURE = 0.2
POSITIVE_OMEGA = 2.0
MOMENTUM_DECAY = 0.9
INPUT_RES = 224
BACKBONES = ('G14', 'B16', 'B32')
CLIP_IDS = {'G14': 'laion/CLIP-ViT-g-14-laion2B-s12B-b42K',
            'B16': 'openai/clip-vit-base-patch16', 'B32': 'openai/clip-vit-base-patch32'}
# Two GPUs: place G14 on GPU1; Qwen+B16+B32 on GPU0. One GPU: leave G14 cuda:0;
# memory is checked by actual execution, with no silent model substitution.
CLIP_DEVICE_MAP = {'G14': 'cuda:1', 'B16': 'cuda:0', 'B32': 'cuda:0'}
ATTACK_DEVICE = 'cuda:0'
QWEN_DEVICE = 'cuda:0'
QWEN_ID = 'Qwen/Qwen3-VL-8B-Instruct'
EVAL_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'  # separate caption VLM, configurable
EVAL_DEVICE = 'cuda:0'
EVALUATE_CAPTIONS = True
MODEL_REVISION_OVERRIDES = {}          # repo_id -> immutable commit; first run resolves once
IMAGE_WEIGHTS = {'G14': 0.6, 'B16': 0.6, 'B32': 0.6}
TEXT_WEIGHTS = {'G14': 0.4, 'B16': 0.4, 'B32': 0.4}
CAPTION_TEMPERATURE = 0.8
CAPTION_TOP_P = 0.9
CAPTION_MAX_NEW_TOKENS = 50            # MPC generate_description
MAX_BANK_ATTEMPTS = 40
QWEN_ATTENTION_LAYER = 29             # SAGA released configuration
QWEN_MAX_PIXELS = None               # use SAGA/processor defaults; attack canvas is224
QWEN_GPU_RESERVE_GIB = 2.0            # execution-only headroom when sharding the8B model
XAI_MAX_SIDE = 448
SAVED_TENSORS_ON_CPU = True           # trade host RAM/PCIe time for attack activation VRAM
LOG_EVERY = 30
# CARA v6.2 additions. These are extensions, not parameters from FINER/SAGA.
NEGATIVE_ATTEMPTS = 8           # retries per hard-negative caption; never pad with duplicates
VALIDATE_PAIR_STEMS = True      # fail fast if source/target file IDs do not align
MASK_UPDATES_TO_CROP = False     # strict region update for hypothesis; False reproduces MPC-style momentum spread
XAI_ON_TARGET = True            # source & target diagnostic heatmaps (step 0)
XAI_FAILURE_POLICY = 'reuse_last'  # only for updates >0; step0 without a valid map fails that sample
ABORT_ON_SAMPLE_ERROR = False   # continue other samples, write failures.json
SAVE_STEP_IMAGES = True        # step_000, 030, 060,...,300 for visualization
RUN_NOTE = 'cara_v6_2'         # output folder name; change when editing experimental method
DATASET_URL = 'https://www.kaggle.com/datasets/zintom/blackbox-adversarial-attack-dataset'


## Package

In [ ]:
# Optional bootstrap. Keep the CUDA-compatible torch/torchvision provided by the host.
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        'transformers==4.57.1', 'accelerate==1.11.0', 'huggingface_hub>=0.34,<1',
        'ftfy==6.3.1', 'qwen-vl-utils==0.0.14', 'regex', 'Pillow', 'numpy', 'pandas', 'matplotlib', 'nbformat',
        'git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6'], check=True)
    print('Dependencies installed. Restart the kernel if transformers/torch was already imported; set INSTALL_DEPENDENCIES=False to avoid reinstalling.')


In [ ]:
import ast, contextlib, gc, hashlib, io, json, math, random, re, sys, time
from collections import Counter
from importlib import metadata as package_metadata
from typing import List
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import datasets
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def free_cuda():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def image_tensor(image):
    return torch.from_numpy(np.array(image, dtype=np.uint8, copy=True)).permute(2,0,1).unsqueeze(0).float()/255

def tensor_image(x):
    a = x.detach().cpu().squeeze(0).clamp(0,1).mul(255).round().byte().permute(1,2,0).numpy()
    return Image.fromarray(a)

def open_rgb(path):
    with Image.open(path) as im: return ImageOps.exif_transpose(im).convert('RGB')

def reduced_image(image, side=XAI_MAX_SIDE):
    image = image.copy(); image.thumbnail((side, side), Image.Resampling.BICUBIC)
    return image

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def load_attack_image(path):
    # MPC ImageFolder preprocessing: Resize shortest edge, then CenterCrop224.
    transform=Compose([Resize(INPUT_RES,interpolation=InterpolationMode.BICUBIC),
                       T.CenterCrop(INPUT_RES)])
    return transform(open_rgb(path))
# Paper author-code definitions below have individual source digests.
from collections.abc import Sequence
import logging, string


## Data

In [ ]:
def sample_count():
    if type(NUM_SAMPLES) is not int or NUM_SAMPLES < 1:
        raise ValueError('NUM_SAMPLES must be a positive integer (e.g. 1, 200, 1000).')
    return NUM_SAMPLES

def prepare_pairs():
    n = sample_count()
    def find_dir(label, override):
        if override is not None:
            p = Path(override)
            if not p.is_dir(): raise FileNotFoundError(p)
            return p
        if not DATASET_ROOT.is_dir(): raise FileNotFoundError(f'Edit DATASET_ROOT: {DATASET_ROOT}')
        found = ([DATASET_ROOT] if DATASET_ROOT.name == label else []) + list(DATASET_ROOT.rglob(label))
        found = [p for p in found if p.is_dir()]
        if len(found) != 1: raise ValueError(f'Expected exactly one {label}: {found}. Set explicit directory.')
        return found[0]
    if PAIR_MANIFEST is not None:
        path = Path(PAIR_MANIFEST)
        pairs = [json.loads(s) for s in path.read_text(encoding='utf-8').splitlines() if s.strip()]
        for row in pairs:
            for role in ('source', 'target'):
                p = Path(row[role]); row[role] = str((p if p.is_absolute() else path.parent/p).resolve())
    else:
        # Preserve ImageFolder sorted class/file pairing from the existing attack notebooks.
        s = datasets.ImageFolder(find_dir('bigscale', SOURCE_DIR))
        t = datasets.ImageFolder(find_dir('target_images', TARGET_DIR))
        if len(s) != len(t): raise ValueError('Unequal source/target count; provide an explicit pair manifest.')
        pairs = [{'index': i, 'source': a[0], 'target': b[0]} for i,(a,b) in enumerate(zip(s.samples,t.samples))]
    if PAIR_MANIFEST is None and VALIDATE_PAIR_STEMS:
        def paired_key(path):
            stem=Path(path).stem.casefold()
            return re.sub(r'[_-](source|target|src|tgt|clean|adv)$','',stem)
        mismatches=[(p['source'],p['target']) for p in pairs if paired_key(p['source'])!=paired_key(p['target'])]
        if mismatches:
            raise ValueError('Source/target filenames have different pair IDs. '
                             'Supply PAIR_MANIFEST, or disable VALIDATE_PAIR_STEMS only after visually checking alignment. '
                             f'First mismatches: {mismatches[:3]}')
    pairs = pairs[SAMPLE_START:SAMPLE_START+n]
    if len(pairs) != n: raise ValueError(f'Need {n} pairs, found {len(pairs)}.')
    indices = [p['index'] for p in pairs]
    if any(type(i) is not int or i < 0 for i in indices) or len(set(indices)) != n:
        raise ValueError('Pair indices must be unique nonnegative integers.')
    for row in pairs:
        if Path(row['source']).resolve() == Path(row['target']).resolve(): raise ValueError('Identical source/target path.')
        row['input_sha256'] = {r: file_hash(row[r]) for r in ('source','target')}
        if row['input_sha256']['source'] == row['input_sha256']['target']:
            raise ValueError('Identical source and target image bytes.')
    return pairs

def validate_config():
    sample_count()
    if RUN_MODE not in {'pilot','full'} or XAI_METHOD not in {'grad_eclip','qwen_attention'}:
        raise ValueError('Choose a known run mode and XAI method.')
    if (STEPS,REFRESH_EVERY,RANDOM_FROM_STEP) != (300,30,150):
        raise ValueError('v6 fixes 300/30/150. Relabel a schedule ablation explicitly.')
    if type(K) is not int or K < 1 or MAX_BANK_ATTEMPTS < K: raise ValueError('Invalid K/attempt budget.')
    if type(SAMPLE_START) is not int or SAMPLE_START < 0: raise ValueError('Invalid SAMPLE_START.')
    if not 0 < TEMPERATURE < float('inf') or not 0 < ALPHA_255 <= EPSILON_255 <= 255:
        raise ValueError('Invalid loss temperature/pixel budget.')
    if EPSILON_255 != int(EPSILON_255): raise ValueError('Use an integer pixel epsilon for exact PNG bounds.')
    if INPUT_RES!=224 or (QWEN_MAX_PIXELS is not None and QWEN_MAX_PIXELS<32*32) or XAI_MAX_SIDE<32:
        raise ValueError('Expected MPC224 preprocessing and valid XAI limits.')
    if not 0<POSITIVE_OMEGA<float('inf') or not 0<=MOMENTUM_DECAY<1:
        raise ValueError('Invalid MPC omega/momentum.')
    if not 0 < CAPTION_TEMPERATURE < float('inf') or not 0 < CAPTION_TOP_P <= 1:
        raise ValueError('Invalid caption sampling.')
    if EVALUATE_CAPTIONS and EVAL_MODEL_ID == QWEN_ID: raise ValueError('Use a distinct evaluation model.')
    for mapping in (CLIP_IDS,CLIP_DEVICE_MAP,IMAGE_WEIGHTS,TEXT_WEIGHTS):
        if set(mapping) != set(BACKBONES): raise ValueError('Expected exactly G14/B16/B32 keys.')
    for name in BACKBONES:
        wi,wt = IMAGE_WEIGHTS[name],TEXT_WEIGHTS[name]
        if not math.isfinite(wi+wt) or min(wi,wt)<0 or wi+wt<=0: raise ValueError('Invalid fusion weights.')
    if not torch.cuda.is_available(): raise RuntimeError('Pretrained execution requires CUDA.')
    for d in list(CLIP_DEVICE_MAP.values()) + [ATTACK_DEVICE,QWEN_DEVICE,EVAL_DEVICE]:
        device = torch.device(d)
        if device.type != 'cuda' or (device.index or 0) >= torch.cuda.device_count():
            raise RuntimeError(f'{d} unavailable. Edit CLIP_DEVICE_MAP/devices for this host.')
    if package_metadata.version('transformers') != '4.57.1':
        raise RuntimeError('Require transformers==4.57.1; run Package install and restart the kernel.')

def resolve_revisions():
    from huggingface_hub import HfApi
    lock_path = OUTPUT_ROOT/'model_revisions.json'
    lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
    ids = list(CLIP_IDS.values()) + [QWEN_ID] + ([EVAL_MODEL_ID] if EVALUATE_CAPTIONS else [])
    for model_id in ids:
        rev = MODEL_REVISION_OVERRIDES.get(model_id, lock.get(model_id))
        if rev is None: rev = HfApi().model_info(model_id).sha
        if not re.fullmatch(r'[a-f0-9]{40}', str(rev)): raise ValueError(f'Expected model commit SHA: {model_id}={rev}')
        lock[model_id] = rev
    atomic_json(lock_path, lock)
    return {name: lock[name] for name in sorted(set(ids))}


## Preprocessing — Original author code (unmodified)

The following cells hold **unaltered functions/prompts** from the pinned author versions in the uploaded v6 notebook. The CARA wrappers are separate. No silent replacement of paper code.

In [ ]:
AUTHOR_SOURCES = {}
AUTHOR_REVISIONS = {'FINER': 'd67831696e122d01c9823114835f7f50fa07b44c', 'MPCAttack': '7b195a5bcbaeaf1e8534d101199810a2782caad3', 'SAGA': '042274599b958d4d63760b9bf44bbcb88493811c', 'Grad-ECLIP': 'e370e6cb194faf2020f5d1ed268f9d57e91a38e6'}


### Author source: `finer_object_prompts.py`

Unmodified function definitions. Source SHA-256: `4834a17db74b9a716865954a98b78fd3dafcaebc2434f1ad78fd2a8336d9fd77`.

In [ ]:
AUTHOR_SOURCES['finer_object_prompts.py'] = r'''SYSTEM_MSG_OBJECTS = (
    "You are an information extraction assistant.\n"
    "From the caption, select up to FIVE main objects. A main object must have at least one descriptive attribute in the caption "
    "(e.g., color, size, material, possession/with-phrase, appositive, relative clause, or an explicit number).\n"
    "Rules:\n"
    "• Output only object names that appear in the caption and are part of the described scene. Never invent or infer.\n"
    "• Prefer plain object names in the output (omit adjectives), but KEEP explicit numbers/quantifiers if the caption states them "
    "(e.g., 'two cats', 'five chickens', 'a pair of skis').\n"
    "• If multiple mentions share the same name and no explicit number is given, output the plural form (e.g., 'dogs'). \n"
    "• List 1–5 main objects; if only one is present, output just that one.\n"
    "Do not add quantifiers like 'some' unless present in the caption.\n"
    "Format:\n"
    "Return EXACTLY one line:\n"
    "PHRASE=<comma-separated list with 'and' before the last item, e.g., 'a dog, a cat and two birds'>\n"
    "No trailing period. No extra text."
)

FEW_SHOT_OBJECTS = [
    (
        "A bustling street scene with three blue buses at the curb, two red cars in traffic, "
        "a tall glass office building behind them, and a bronze statue with a raised arm in the plaza.",
        "PHRASE=three buses, two cars, an office building and a statue"
    ),
    (
        "On a wooden table, a cast-iron skillet sits beside a blue ceramic vase of sunflowers, "
        "with red apples and green pears piled high in a stainless steel bowl.",
        "PHRASE=a skillet, a vase, apples, pears and a bowl"
    ),
]

SYSTEM_MSG_NEGOBJ = (
    "You are a negative object creator.\n"
    "You will receive a caption, an object list as PHRASE=..., and REPLACE_INDEX=k (1-based).\n"
    "Replace EXACTLY the k-th object in PHRASE with a distinctly different NEGATIVE object.\n"
    "\n"
    "Keep ALL other objects unchanged and preserve their order and punctuation. Keep the same quantifier/determiner "
    "for the replaced slot (e.g., 'two cats' -> 'two bicycles').\n"
    "\n"
    "Constraints for the NEGATIVE object:\n"
    "• It must be distinctly different from the replaced object (not a synonym; not just singular/plural).\n"
    "• It must NOT be a synonym or near-equivalent of ANY object that appears in the caption.\n"
    "• It must NOT appear anywhere in the caption (as a whole word, singular or plural).\n"
    "• Do not modify any other items; do not reorder items; do not add or remove items.\n"
    "\n"
    "Edge cases (must follow):\n"
    "• If REPLACE_INDEX is greater than the number of objects in PHRASE, replace the LAST object.\n"
    "• If REPLACE_INDEX is less than 1, replace the FIRST object.\n"
    "\n"
    "Self-check (must hold):\n"
    "• Same number of items as input; exactly one item (the k-th per the rule above) differs.\n"
    "\n"
    "Output EXACTLY one line:\n"
    "PHRASE=<the new list, same format>\n"
    "No extra text. No quotes. No trailing period."
)

# Few-shots cover different list lengths and positions; each uses REPLACE_INDEX to avoid a last-item bias.
FEW_SHOT_NEGOBJ = [
    (
        # Replace FIRST of three
        "Caption:\nA small yellow school bus is parked next to two red cars in front of a brick office building.\n"
        "PHRASE=a bus, two cars and an office building\n"
        "REPLACE_INDEX=1",
        "PHRASE=a boat, two cars and an office building"
    ),
    (
        "Caption:\nTwo hikers stand on a wooden bridge with their dog beside them over a calm river.\n"
        "PHRASE=two hikers, a dog, a bridge and a river\n"
        "REPLACE_INDEX=2",
        "PHRASE=two hikers, a bicycle, a bridge and a river"
    ),
    (
        "Caption:\nOn a side table there is a bowl of fruit, a ceramic vase, and a brass lamp.\n"
        "PHRASE=a bowl of fruit, a vase and a lamp\n"
        "REPLACE_INDEX=3",
        "PHRASE=a bowl of fruit, a vase and a clock"
    ),
    (
        "Caption:\nA dashboard card labeled 'Latency' shows a graph with a menu icon and a legend for several lines.\n"
        "PHRASE=a card, a graph, a menu icon, a legend and lines\n"
        "REPLACE_INDEX=3",
        "PHRASE=a card, a graph, a slider, a legend and lines"
    ),
    (
        "Caption:\nA booking screen shows a highlighted 12:05–12:30 time slot, a search box, an entry for Animal Kingdom Villas, a restaurant icon, and a market link.\n"
        "PHRASE=a time slot, a box, an entry, a restaurant and a market\n"
        "REPLACE_INDEX=4",
        "PHRASE=a time slot, a box, an entry, a ticket counter and a market"
    ),
]

CATEGORIES = ["natural_image", "chart_graph", "screenshot_ui", "document_text"]

SYSTEM_MSG_CATEGORY = (
    "You are a strict classifier. Based ONLY on the caption text, assign EXACTLY ONE label.\n"
    "Allowed labels:\n"
    "  - natural_image\n"
    "  - chart_graph   (includes diagrams, flowcharts, schematics, illustrations)\n"
    "  - screenshot_ui (UI/app/website/IDE/terminal screenshots)\n"
    "  - document_text (scans, printed pages, receipts, invoices, whiteboards, handwriting)\n"
    "\n"
    "Rules:\n"
    "• Return ONLY the label string, nothing else.\n"
    "• If multiple seem plausible, pick the single best one.\n"
    "• If unsure, choose natural_image.\n"
)

FEW_SHOT_CATEGORY = [
    ("A line chart comparing quarterly revenue for three products with a legend and axes labels.",
     "chart_graph"),
    ("A screenshot of a mobile banking app showing the account balance and a transfer button.",
     "screenshot_ui"),
    ("A scanned receipt from a grocery store listing items and totals.",
     "document_text"),
    ("A photo of a dog running on a beach at sunset.",
     "natural_image"),
    ("A flowchart illustrating an authentication pipeline with decision nodes and arrows.",
     "chart_graph"),
]

'''


### Author source: `finer_general_helpers.py`

Unmodified function definitions. Source SHA-256: `5f8970a2ee394b75eb8640ccea43bb69764ab6b3f858cbed903f0f9f7cd3b5e4`.

In [ ]:
AUTHOR_SOURCES['finer_general_helpers.py'] = r'''def truncate_first_tokens(tokenizer, text: str, max_tokens: int = 512) -> str:
    """
    Keep only the first `max_tokens` tokenizer tokens.
    """
    ids = tokenizer(text or "", add_special_tokens=False).input_ids
    ids = ids[:max_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def parse_tagged_output(
    text: str,
    *,
    allowed_tags: Sequence[str],
    max_chars: int = 512,
    split_on_semicolon: bool = False,
    strip_trailing_pattern: str = r"[.;:\s]+$",
) -> str:
    """
    Parse a generated single-line output like:
      PHRASE=...
      SENTENCE=...
      ACC_PHRASE=...
    by scanning for the first matching tagged line.

    Fallback:
      - if no tagged line is found, use the first non-empty line.

    Normalization:
      - strip outer quotes
      - optional split on first ';'
      - strip trailing punctuation via regex
      - collapse internal whitespace
      - clip to max_chars
    """
    raw = (text or "").strip()
    lines = [
        line.strip()
        for line in raw.splitlines()
        if line.strip() and line.strip() not in {"```", "~~~"}
    ]

    if not lines:
        return ""

    tag_pattern = "|".join(re.escape(tag) for tag in allowed_tags)
    pattern = re.compile(
        rf"^\s*(?:{tag_pattern})\s*[:=]\s*(.+)$",
        flags=re.IGNORECASE,
    )

    value = ""
    for line in lines:
        match = pattern.match(line)
        if match:
            value = match.group(1).strip()
            break

    if not value:
        value = lines[0].strip()

    value = (
        value.strip()
        .strip('"')
        .strip("'")
        .strip("“”‘’")
        .strip()
    )

    if split_on_semicolon:
        value = value.split(";", 1)[0].strip()

    if strip_trailing_pattern:
        value = re.sub(strip_trailing_pattern, "", value)

    value = re.sub(r"\s+", " ", value).strip()

    if max_chars is not None:
        value = value[:max_chars]

    return value'''


### Author source: `finer_object_helpers.py`

Unmodified function definitions. Source SHA-256: `97fe84f6b421ef8e8c7a3cbe1aa3551911d50ac6931d7165a8b8da9bb520e9be`.

In [ ]:
AUTHOR_SOURCES['finer_object_helpers.py'] = r'''def _split_objects_phrase(phrase: str) -> List[str]:
    """
    Split 'a dog, a cat and two birds' -> ['a dog','a cat','two birds'].
    Heuristic: split on the last ' and ' and commas on the left.
    Used only for lightweight validation; storage remains a single string.
    """
    s = (phrase or "").strip()
    if not s:
        return []
    if " and " in s:
        left, last = s.rsplit(" and ", 1)
        left_parts = [p.strip() for p in left.split(",") if p.strip()]
        items = left_parts + [last.strip()]
    else:
        items = [p.strip() for p in s.split(",") if p.strip()]
    return items

def _messages_objects(caption_text: str) -> List[dict]:
    """Chat messages for the 'objects' mode with few-shot."""
    msgs = [{"role": "system", "content": SYSTEM_MSG_OBJECTS}]

    for cap, ans in FEW_SHOT_OBJECTS:
        demo_user = (
            "Caption:\n" + cap + "\n\n"
            "Task: From this caption, select up to five main objects that have at least one descriptive attribute in the caption. "
            "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "
            "Return EXACTLY one line:\nPHRASE=<...>"
        )
        msgs.append({"role": "user", "content": demo_user})
        msgs.append({"role": "assistant", "content": ans})

    task_user = (
        "Caption:\n" + caption_text + "\n\n"
        "Task: Select up to five main objects that have at least one descriptive attribute in the caption. "
        "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "
        "Return EXACTLY one line:\nPHRASE=<...>"
    )
    msgs.append({"role": "user", "content": task_user})
    return msgs

def build_objects_prompt(tokenizer, caption: str) -> str:
    """Final chat prompt using the FIRST 512 caption tokens."""
    cap_trunc = _truncate_caption_first_tokens(tokenizer, caption, 512)
    return tokenizer.apply_chat_template(
        _messages_objects(cap_trunc),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

def parse_objects_output(text: str) -> str:
    return parse_tagged_output(
        text,
        allowed_tags=("PHRASE",),
        max_chars=512,
    )'''


### Author source: `saga_attention_utils.py`

Unmodified function definitions. Source SHA-256: `af68ce21bded5d34e138c7059f313daa34d5a8d055082a940cb0e34a011f4983`.

In [ ]:
AUTHOR_SOURCES['saga_attention_utils.py'] = r'''def get_token_indices(decoded_text: str, tokenizer, outputs: dict, inputs: dict, vlm_type="qwen3_vl") -> dict:
    """Extract indices for vision tokens, prompt tokens, and output tokens."""
    if vlm_type == "qwen3_vl":
        marker_start = "<|vision_start|>"
        marker_end = "<|vision_end|>"

    len_prompt_tokens = len(inputs["input_ids"][0])
    len_all_tokens = len(outputs["sequences"][0])

    # +1 to include <|vision_start|> token
    vision_token_start = (
        len(tokenizer(decoded_text.split(marker_start)[0], return_tensors="pt")["input_ids"][0]) + 1
    )
    vision_token_end = len(tokenizer(decoded_text.split(marker_end)[0], return_tensors="pt")["input_ids"][0])

    return {
        "vision_start": vision_token_start,
        "vision_end": vision_token_end,
        "output_start": len_prompt_tokens,
        "output_end": len_all_tokens,
        "output_len": len_all_tokens - len_prompt_tokens,
    }

def aggregate_llm_attention(attn: Tensor, layer_idx: int | None = None) -> Tensor:
    """
    Extract average attention vector from attention outputs.

    Nullifies attention over the first token (<bos>), which is a common practice
    for null attention (https://aclanthology.org/W19-4808.pdf).

    Args:
        attn: Attention tensor from model outputs
        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.
    """
    if layer_idx is not None and layer_idx != -100:
        layer = attn[layer_idx]
        layer_attns = layer.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        vec = torch.concat(
            [
                torch.tensor([0.0]),  # Zero out null attention on <bos>
                attns_per_head[-1][1:].cpu(),
                torch.tensor([0.0]),  # Zero for final generated token
            ]
        )
        return vec / vec.sum()

    avged = []
    for layer in attn:
        layer_attns = layer.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        vec = torch.concat(
            [
                torch.tensor([0.0]),  # Zero out null attention on <bos>
                attns_per_head[-1][1:].cpu(),
                torch.tensor([0.0]),  # Zero for final generated token
            ]
        )
        avged.append(vec / vec.sum())
    return torch.stack(avged).mean(dim=0)

def heterogenous_stack(vecs: list[torch.Tensor]) -> torch.Tensor:
    """Pad vectors with zeros to same length then stack."""
    max_length = max(v.shape[0] for v in vecs)
    return torch.stack([torch.concat((v, torch.zeros(max_length - v.shape[0]))) for v in vecs])

def aggregate_prompt_attention(
    first_generated_token_attn: Tensor,
    layer_idx: int | None = None,
) -> Tensor:
    """
    Aggregate attention for the prompt/first generated token.

    Args:
        first_generated_token_attn: Attention tensor from first generation step
        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.
    """
    if layer_idx is not None and layer_idx != -100:
        layer_attn = first_generated_token_attn[layer_idx]
        layer_attns = layer_attn.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        # last row corresponds to the generated token, exclude it
        cur = attns_per_head[:-1].cpu().clone()

        # Zero out attention to the first <bos> token (null attention)
        # Skip first row since <bos> is the only token it can attend to
        cur[1:, 0] = 0.0
        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)

        return cur

    aggregated = []
    for layer_attn in first_generated_token_attn:
        layer_attns = layer_attn.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        # last row corresponds to the generated token, exclude it
        cur = attns_per_head[:-1].cpu().clone()

        # Zero out attention to the first <bos> token (null attention)
        # Skip first row since <bos> is the only token it can attend to
        cur[1:, 0] = 0.0
        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)

        aggregated.append(cur)

    # Average across all layers
    return torch.stack(aggregated).mean(dim=0)

def build_attention_matrix(outputs, layer_idx: int | None = None, valid_token_indices=None) -> torch.Tensor:
    """
    Build a complete attention matrix from generation outputs.

    Args:
        outputs: Model generation outputs with attentions
        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.
        valid_token_indices: Optional set of valid token indices to include in the attention matrix.

    Returns:
        Tensor of shape (total_tokens, max_seq_len) representing attention
        across all tokens (prompt + generated)
        entry (i, j): how much token i attends to token j
    """
    attention_vectors = []

    # Add <bos> token (attends only to itself)
    attention_vectors.append(torch.tensor([1.0]))

    # Add prompt token attentions
    prompt_attention = aggregate_prompt_attention(outputs.attentions[0], layer_idx=layer_idx)
    attention_vectors.extend(list(prompt_attention))

    # Add generated token attentions
    for step_idx, step_attention in enumerate(outputs.attentions):
        if valid_token_indices is not None and step_idx not in valid_token_indices:
            continue
        gen_attention = aggregate_llm_attention(step_attention, layer_idx=layer_idx)
        attention_vectors.append(gen_attention)

    # Pad all vectors to same length and stack
    return heterogenous_stack(attention_vectors)

def get_token_img_shape(inputs, processor, vlm_type="qwen3_vl"):
    if vlm_type == "qwen3_vl":
        """
        Qwen3-VL: Calculate the grid dimensions (height, width) of image tokens after spatial merging.
        NOTE: Processor outputs containing 'image_grid_thw'.
        """
        # Get the raw grid from the processor inputs
        grid_h = inputs["image_grid_thw"][0][1].item()
        grid_w = inputs["image_grid_thw"][0][2].item()

        # 2x2 spatial merging/pooling
        merge_factor = processor.image_processor.merge_size
        grid_h = grid_h // merge_factor
        grid_w = grid_w // merge_factor
        return grid_h, grid_w'''


### Author source: `saga_text.py`

Unmodified function definitions. Source SHA-256: `8205a79ed5fe0d1588091e6520a00ef9941fa3441f691cd0fcb92c5a26b50abc`.

In [ ]:
AUTHOR_SOURCES['saga_text.py'] = r'''STOP_WORDS = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "you're",
    "you've",
    "you'll",
    "you'd",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "he",
    "him",
    "his",
    "himself",
    "she",
    "she's",
    "her",
    "hers",
    "herself",
    "it",
    "it's",
    "its",
    "itself",
    "they",
    "them",
    "their",
    "theirs",
    "themselves",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "that'll",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "s",
    "t",
    "can",
    "will",
    "just",
    "don",
    "don't",
    "should",
    "should've",
    "now",
    "d",
    "ll",
    "m",
    "o",
    "re",
    "ve",
    "y",
    "ain",
    "aren",
    "aren't",
    "couldn",
    "couldn't",
    "didn",
    "didn't",
    "doesn",
    "doesn't",
    "hadn",
    "hadn't",
    "hasn",
    "hasn't",
    "haven",
    "haven't",
    "isn",
    "isn't",
    "ma",
    "mightn",
    "mightn't",
    "mustn",
    "mustn't",
    "needn",
    "needn't",
    "shan",
    "shan't",
    "shouldn",
    "shouldn't",
    "wasn",
    "wasn't",
    "weren",
    "weren't",
    "won",
    "won't",
    "wouldn",
    "wouldn't",
}

def clean_token(token: str) -> str:
    """Lowercase, strip, and drop punctuation to normalize tokens."""
    return token.strip().lower().translate(str.maketrans("", "", string.punctuation))

def is_content_word(token: str) -> bool:
    """Return True if token is non-empty after cleaning and not a stop word."""
    cleaned = clean_token(token)
    return bool(cleaned) and cleaned not in STOP_WORDS

def filter_generated_tokens(tokenizer, token_ids: Sequence[int], use_content_words: bool) -> list[int]:
    """
    Return indices of tokens to use. If use_content_words is True, keep only content
    words; otherwise return all indices. Falls back to all tokens if filtering empties.
    """
    if not use_content_words:
        return list(range(len(token_ids)))

    logger.debug(f"[Use content words: {use_content_words}] Only keeping content words")
    valid_indices = []
    for idx, token_id in enumerate(token_ids):
        token_str = tokenizer.decode([token_id], skip_special_tokens=False)
        if is_content_word(token_str):
            valid_indices.append(idx)
        elif DEBUG:
            logger.debug(f"Filtering out token '{token_str}' at index {idx}")

    if not valid_indices:
        return list(range(len(token_ids)))

    return valid_indices'''


### Author source: `saga_attention_map.py`

Unmodified function definitions. Source SHA-256: `e408b4168e02c2f2ef0e0d41f9284295a66e546b7067a80d049b2d947711dee7`.

In [ ]:
AUTHOR_SOURCES['saga_attention_map.py'] = r'''def get_attention_map_from_generated_text_vlm(
    model,
    processor,
    inputs,
    vlm_type: str,
    use_vis_attention: bool = False,
    temperature: float = 1.0,
    do_sample: bool = False,
    layer_idx: int | None = None,
    use_content_words: bool = False,
    norm_mode: str = "minmax",
    **kwargs,
) -> tuple[Tensor, str]:
    """Extract generation-time Qwen3-VL attention using the original causal row alignment."""

    if vlm_type != "qwen3_vl":
        raise ValueError("This release supports Qwen3-VL only.")

    tokenizer = processor.tokenizer

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        return_dict_in_generate=True,
        output_attentions=True,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
    )

    decoded_text = processor.batch_decode(
        outputs["sequences"], skip_special_tokens=False, clean_up_tokenization_spaces=False
    )[0]

    token_indices = get_token_indices(decoded_text, tokenizer, outputs, inputs, vlm_type=vlm_type)
    generated_token_ids = outputs.sequences[0][token_indices["output_start"] : token_indices["output_end"]]
    valid_token_indices = filter_generated_tokens(tokenizer, generated_token_ids, use_content_words)

    # Build attention matrix and get token indices
    # Keep every generation step: compacting filtered rows breaks causal row alignment.
    llm_attn_matrix = build_attention_matrix(outputs=outputs, layer_idx=layer_idx)

    vision_token_start = token_indices["vision_start"]
    vision_token_end = token_indices["vision_end"]
    num_vision_tokens = vision_token_end - vision_token_start

    grid_h, grid_w = get_token_img_shape(inputs=inputs, processor=processor, vlm_type=vlm_type)
    assert grid_h * grid_w == num_vision_tokens, (
        f"Mismatch in vision tokens ({num_vision_tokens}) and grid size: {num_vision_tokens} vs {grid_h}x{grid_w}"
    )

    # Average attention maps over all valid generated tokens
    attn_maps = []
    for token_offset in valid_token_indices:
        token_row = token_indices["output_start"] + token_offset
        if token_row >= llm_attn_matrix.shape[0]:
            logger.warning(
                f"Token row {token_row} out of range for attention matrix with shape {llm_attn_matrix.shape}"
            )
            continue

        attn_weights_over_vis_tokens = llm_attn_matrix[token_row][vision_token_start:vision_token_end]
        if attn_weights_over_vis_tokens.sum() == 0:
            continue
        attn_weights_over_vis_tokens = attn_weights_over_vis_tokens / attn_weights_over_vis_tokens.sum()
        attn_maps.append(attn_weights_over_vis_tokens.reshape(grid_h, grid_w))

    if attn_maps:
        attn_over_image = torch.stack(attn_maps).mean(dim=0)
    else:
        attn_over_image = torch.zeros((grid_h, grid_w), dtype=llm_attn_matrix.dtype)

    mn = attn_over_image.min()
    mx = attn_over_image.max()
    attn_over_image = (attn_over_image - mn) / (mx - mn + 1e-8)
    logger.debug(f"Attention map shape: {attn_over_image.shape}")

    generated_text = tokenizer.decode(
        outputs.sequences[0][token_indices["output_start"] : token_indices["output_end"]],
        skip_special_tokens=True,
    )

    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True)
    return attn_over_image, generated_text'''


### Author source: `gradeclip_author_functions.py`

Unmodified function definitions. Source SHA-256: `41d72e9c60c771955be9460b50132c3bfe928e68033e68cbe3bc8b01bcec6838`.

In [ ]:
AUTHOR_SOURCES['gradeclip_author_functions.py'] = r'''def attention_layer(q, k, v, num_heads=1):
    "Compute 'Scaled Dot Product Attention'"
    tgt_len, bsz, embed_dim = q.shape
    head_dim = embed_dim // num_heads
    scaling = float(head_dim) ** -0.5
    q = q * scaling
    
    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)
    k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    v = v.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    attn_output_weights = torch.bmm(q, k.transpose(1, 2))
    attn_output_weights = F.softmax(attn_output_weights, dim=-1)
    attn_output_heads = torch.bmm(attn_output_weights, v)
    assert list(attn_output_heads.size()) == [bsz * num_heads, tgt_len, head_dim]
    attn_output = attn_output_heads.transpose(0, 1).contiguous().view(tgt_len, bsz, embed_dim)
    attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, -1)
    attn_output_weights = attn_output_weights.sum(dim=1) / num_heads
    return attn_output, attn_output_weights

def clip_encode_dense(x,n):
    vision_width = clipmodel.visual.transformer.width
    vision_heads = vision_width // 64
    print("[vision_width and vision_heads]:", vision_width, vision_heads)
    
    # modified from CLIP
    x = x.half()
    x = clipmodel.visual.conv1(x)  
    feah, feaw = x.shape[-2:]

    x = x.reshape(x.shape[0], x.shape[1], -1) 
    x = x.permute(0, 2, 1) 
    class_embedding = clipmodel.visual.class_embedding.to(x.dtype)
    x = torch.cat([class_embedding + torch.zeros(x.shape[0], 1, x.shape[-1]).to(x), x], dim=1)

    ## scale position embedding as the image w-h ratio
    pos_embedding = clipmodel.visual.positional_embedding.to(x.dtype)
    tok_pos, img_pos = pos_embedding[:1, :], pos_embedding[1:, :]
    pos_h = clip_inres // clip_ksize[0]
    pos_w = clip_inres // clip_ksize[1]
    assert img_pos.size(0) == (pos_h * pos_w), f"the size of pos_embedding ({img_pos.size(0)}) does not match resolution shape pos_h ({pos_h}) * pos_w ({pos_w})"
    img_pos = img_pos.reshape(1, pos_h, pos_w, img_pos.shape[1]).permute(0, 3, 1, 2)
    print("[POS shape]:", img_pos.shape, (feah, feaw))
    img_pos = torch.nn.functional.interpolate(img_pos, size=(feah, feaw), mode='bicubic', align_corners=False)
    img_pos = img_pos.reshape(1, img_pos.shape[1], -1).permute(0, 2, 1)
    pos_embedding = torch.cat((tok_pos[None, ...], img_pos), dim=1)
    x = x + pos_embedding
    x = clipmodel.visual.ln_pre(x)
    
    x = x.permute(1, 0, 2)  # NLD -> LND
    x = torch.nn.Sequential(*clipmodel.visual.transformer.resblocks[:-n])(x)

    attns = []
    atten_outs = []
    vs = []
    qs = []
    ks = []
    for TR in clipmodel.visual.transformer.resblocks[-n:]:
        x_in = x
        x = TR.ln_1(x_in)
        linear = torch._C._nn.linear    
        q, k, v = linear(x, TR.attn.in_proj_weight, TR.attn.in_proj_bias).chunk(3, dim=-1)
        attn_output, attn = attention_layer(q, k, v, 1)  # vision_heads=1
        attns.append(attn)
        atten_outs.append(attn_output)
        vs.append(v)
        qs.append(q)
        ks.append(k)
        
        x_after_attn = linear(attn_output, TR.attn.out_proj.weight, TR.attn.out_proj.bias)       
        x = x_after_attn + x_in
        x = x + TR.mlp(TR.ln_2(x))

    x = x.permute(1, 0, 2)  # LND -> NLD
    x = clipmodel.visual.ln_post(x)
    x = x @ clipmodel.visual.proj
    return x, x_in, vs, qs, ks, attns, atten_outs, (feah, feaw)

def sim_qk(q, k):
    q_cls = F.normalize(q[:1,0,:], dim=-1) 
    k_patch = F.normalize(k[1:,0,:], dim=-1)

    cosine_qk = (q_cls * k_patch).sum(-1) 
    cosine_qk_max = cosine_qk.max(dim=-1, keepdim=True)[0]
    cosine_qk_min = cosine_qk.min(dim=-1, keepdim=True)[0]
    cosine_qk = (cosine_qk-cosine_qk_min) / (cosine_qk_max-cosine_qk_min)
    return cosine_qk

def grad_eclip(c, qs, ks, vs, attn_outputs, map_size):
    ## gradient on last attention output
    tmp_maps = []
    for q, k, v, attn_output in zip(qs, ks, vs, attn_outputs):
        grad = torch.autograd.grad(
            c,
            attn_output,
            retain_graph=True)[0]

        grad_cls = grad[:1,0,:]
        v_patch = v[1:,0,:]
        cosine_qk = sim_qk(q, k).reshape(-1)
        tmp_maps.append((grad_cls * v_patch * cosine_qk[:,None]).sum(-1)) 

    emap = F.relu_(torch.stack(tmp_maps, dim=0)).sum(0)
    return emap.reshape(*map_size)'''


### Author source: `mpc_info_nce_loss.py`

Unmodified function definitions. Source SHA-256: `857c91ff8e6fe6290a3e569135b97cc2c2898c55a22347a294a184e068ee804c`.

In [ ]:
AUTHOR_SOURCES['mpc_info_nce_loss.py'] = r'''def info_nce_loss(feat_adv, feat_tgt, feat_src, temperature=0.2, mode='img', omega=2.0):
    """
    InfoNCE loss for pulling adv->tgt close, pushing adv->src away.

    Args:
        feat_adv: Tensor [B, D] adversarial features
        feat_tgt: Tensor [B, D] target features (positive)
        feat_src: Tensor [B, D] source features (negative)
        temperature: float, softmax temperature

    Returns:
        loss: scalar tensor
    """
    # L2 normalize
    feat_adv = F.normalize(feat_adv, dim=-1)
    feat_tgt = F.normalize(feat_tgt, dim=-1)
    feat_src = F.normalize(feat_src, dim=-1)
    # Cosine similarities
    if mode == 'text':
        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1).mean(dim=1) * omega / temperature  # [B]
        sim_neg = torch.sum(feat_adv * feat_src, dim=-1).mean(dim=1) / temperature  # [B]
    else:
        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1) * omega / temperature  # [B]
        sim_neg = torch.sum(feat_adv * feat_src, dim=-1) / temperature  # [B]
    # Stack positive and negative: shape [B, 2]
    logits = torch.stack([sim_pos, sim_neg], dim=1)
    labels = torch.zeros(feat_adv.size(0), dtype=torch.long, device=feat_adv.device)  # positives index=0
    # Cross-entropy loss
    loss = F.cross_entropy(logits, labels)
    return - loss'''


### Author source: FINER `object_utils.py` — negative-object functions

Functions below come from FINER `train/generate_training_data/utils/object_utils.py` (same commit as above), and are stored as original function definitions. They generate **negative object phrases**, not complete negative captions.

In [ ]:
AUTHOR_SOURCES['finer_negative_helpers.py'] = r'''def _validate_single_replacement(original_phrase: str, new_phrase: str) -> bool:
    """
    Light sanity: same number of items and exactly one item differs (string-compare after strip).
    """
    old = _split_objects_phrase(original_phrase)
    new = _split_objects_phrase(new_phrase)
    if not old or not new or len(old) != len(new):
        return False
    diffs = sum(1 for a, b in zip(old, new) if a.strip() != b.strip())
    return diffs == 1


def _messages_negobj(caption_text: str, obj_phrase: str, replace_index_1based: int) -> List[dict]:
    """
    Chat messages for negative replacement with an explicit slot to replace.
    The model must replace exactly REPLACE_INDEX (1-based).
    """
    msgs = [{"role": "system", "content": SYSTEM_MSG_NEGOBJ}]
    for demo_user, demo_assistant in FEW_SHOT_NEGOBJ:
        msgs.append({"role": "user", "content": demo_user})
        msgs.append({"role": "assistant", "content": demo_assistant})
    task_user = f"Caption:\n{caption_text}\nPHRASE={obj_phrase}\nREPLACE_INDEX={replace_index_1based}"
    msgs.append({"role": "user", "content": task_user})
    return msgs


def build_negobj_prompt(tokenizer, caption: str, obj_phrase: str, replace_index_1based: int) -> str:
    """Final chat prompt for negative replacement, with an explicit index (1-based) to replace."""
    cap_trunc = _truncate_caption_first_tokens(tokenizer, caption, 512)
    return tokenizer.apply_chat_template(
        _messages_negobj(cap_trunc, obj_phrase, replace_index_1based),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )'''


In [ ]:
AUTHOR_SHA256 = {name: hashlib.sha256(src.encode('utf-8')).hexdigest() for name,src in AUTHOR_SOURCES.items()}
# Verify against the v6 file's verified original author hashes.
EXPECTED_AUTHOR_SHA256 = {'finer_object_prompts.py': '4834a17db74b9a716865954a98b78fd3dafcaebc2434f1ad78fd2a8336d9fd77', 'finer_general_helpers.py': '5f8970a2ee394b75eb8640ccea43bb69764ab6b3f858cbed903f0f9f7cd3b5e4', 'finer_object_helpers.py': '97fe84f6b421ef8e8c7a3cbe1aa3551911d50ac6931d7165a8b8da9bb520e9be', 'saga_attention_utils.py': 'af68ce21bded5d34e138c7059f313daa34d5a8d055082a940cb0e34a011f4983', 'saga_text.py': '8205a79ed5fe0d1588091e6520a00ef9941fa3441f691cd0fcb92c5a26b50abc', 'saga_attention_map.py': 'e408b4168e02c2f2ef0e0d41f9284295a66e546b7067a80d049b2d947711dee7', 'gradeclip_author_functions.py': '41d72e9c60c771955be9460b50132c3bfe928e68033e68cbe3bc8b01bcec6838', 'mpc_info_nce_loss.py': '857c91ff8e6fe6290a3e569135b97cc2c2898c55a22347a294a184e068ee804c'}
for name, expected in EXPECTED_AUTHOR_SHA256.items():
    assert AUTHOR_SHA256[name] == expected, f'Original author source changed: {name}'
for source in AUTHOR_SOURCES.values():
    ast.parse(source)
FINER_NS = {'List':List, 'Sequence':Sequence, 're':re}
for f in ('finer_object_prompts.py','finer_general_helpers.py'):
    exec(compile(AUTHOR_SOURCES[f], f'<{f}>', 'exec'), FINER_NS)
FINER_NS['_truncate_caption_first_tokens'] = FINER_NS['truncate_first_tokens']
for f in ('finer_object_helpers.py','finer_negative_helpers.py'):
    exec(compile(AUTHOR_SOURCES[f], f'<{f}>', 'exec'), FINER_NS)
MPC_NS = {'torch':torch, 'F':F}
exec(compile(AUTHOR_SOURCES['mpc_info_nce_loss.py'], '<mpc-author>', 'exec'), MPC_NS)
SAGA_NS = {'torch':torch, 'Tensor':torch.Tensor, 'Sequence':Sequence, 'string':string,
           'logger':logging.getLogger('cara.saga'), 'DEBUG':False}
for f in ('saga_attention_utils.py','saga_text.py','saga_attention_map.py'):
    exec(compile(AUTHOR_SOURCES[f], f'<{f}>', 'exec'), SAGA_NS)
print('Author source SHA-256 checked:', len(EXPECTED_AUTHOR_SHA256), 'files; FINER negative helpers loaded.')


### CARA preprocessing — concepts and negative captions

The upstream FINER function replaces an object **in `PHRASE=`**. The deterministic mapping from that output into the original caption is explicitly a **CARA integration**, not part of FINER. A candidate is rejected if one-object validation or in-caption alignment fails (no invented fallback caption).

In [ ]:
def replace_object_in_caption(caption, old_item, new_item):
    """CARA extension: replace one verified object span without changing other caption text."""
    # Remove shared determiner/count prefix; change the object noun phrase itself.
    old_words = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", old_item)
    new_words = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", new_item)
    common = 0
    while common < min(len(old_words), len(new_words)) and old_words[common].lower() == new_words[common].lower():
        common += 1
    old_core = old_words[common:]
    new_core = new_words[common:]
    if not old_core or not new_core:
        return None
    # Try longer source-object spans before a shorter noun head.
    for start in range(len(old_core)):
        phrase = ' '.join(old_core[start:])
        replacement = ' '.join(new_core[start:]) if len(new_core)>start else ' '.join(new_core)
        if not replacement: continue
        pattern = re.compile(r'\b' + re.escape(phrase).replace(r'\ ', r'\s+') + r'\b', re.I)
        hits = list(pattern.finditer(caption))
        if len(hits) == 1:
            result = caption[:hits[0].start()] + replacement + caption[hits[0].end():]
            return result if result.casefold()!=caption.casefold() else None
    return None


def validate_negative_phrase(source_phrase, negative_phrase, position):
    old = FINER_NS['_split_objects_phrase'](source_phrase)
    new = FINER_NS['_split_objects_phrase'](negative_phrase)
    return (FINER_NS['_validate_single_replacement'](source_phrase,negative_phrase)
            and len(old)==len(new) and 1<=position<=len(old)
            and all(i==position-1 or old[i]==new[i] for i in range(len(old))))


def negative_caption_bank(captioner, source_bank, target_captions=()):
    """5 distinct, validated source-object counterfactual captions; no duplicate padding."""
    records, negatives, seen = [], [], set()
    for source_caption, obj in zip(source_bank['captions'], source_bank['object_records']):
        source_phrase = obj['phrase']; options=FINER_NS['_split_objects_phrase'](source_phrase)
        attempts=[]; accepted=None
        for _ in range(NEGATIVE_ATTEMPTS):
            position=random.randint(1,len(options))  # original FINER samples a uniform slot
            prompt=FINER_NS['build_negobj_prompt'](captioner.processor.tokenizer,
                                                    source_caption,source_phrase,position)
            raw=captioner.generate(prompt,max_new_tokens=64)
            neg_phrase=FINER_NS['parse_objects_output'](raw)
            valid=validate_negative_phrase(source_phrase,neg_phrase,position)
            candidate=None
            if valid:
                new_options=FINER_NS['_split_objects_phrase'](neg_phrase)
                candidate=replace_object_in_caption(source_caption,options[position-1],new_options[position-1])
            # Reject repeated captions, no-op replacements, or impossible lexical mapping.
            ok=bool(candidate and candidate.casefold() not in seen
                    and candidate.casefold()!=source_caption.casefold()
                    and candidate.casefold() not in {t.casefold() for t in target_captions})
            attempts.append({'position':position,'negative_phrase':neg_phrase,
                             'validated_phrase':bool(valid),'reconstructed_caption':candidate,'accepted':ok})
            if ok:
                seen.add(candidate.casefold());negatives.append(candidate)
                accepted={'source_caption':source_caption,'source_phrase':source_phrase,
                          'negative_phrase':neg_phrase,'replace_index':position,
                          'negative_caption':candidate,'attempts':attempts}
                break
        if accepted is None:
            raise RuntimeError(f'Cannot form validated negative caption for: {source_caption!r}. '
                               f'Inspect attempts={attempts}')
        records.append(accepted)
    assert len(negatives)==K and len(set(negatives))==K
    return {'negative_captions':negatives,'negative_records':records}


### CARA preprocessing — XAI crop, dispersion, and safe snapshots

In [ ]:
def crop_area(step,steps=300,refresh_every=30):
    if (steps,refresh_every)!=(300,30) or type(step) is not int or not 1<=step<=steps:
        raise ValueError('Expected300 updates,30-update stages and an integer step in[1,300].')
    phase=(step-1)//refresh_every
    return (9+phase)/18  #10 stages:0.5,...,1.0 inclusive

def crop_mode(step, random_from=150):
    return 'random' if step >= random_from else 'xai'

def select_box(heatmap, area, mode, generator):
    heat = torch.as_tensor(heatmap).detach().float().cpu()
    if heat.ndim != 2 or not torch.isfinite(heat).all() or not 0 < area <= 1:
        raise ValueError('Invalid spatial map/area.')
    H, W = heat.shape
    h = max(1, min(H, int(round(H*math.sqrt(area)))))
    w = max(1, min(W, int(round(W*math.sqrt(area)))))
    if area == 1: return (0, 0, H, W)
    if mode == 'random':
        y = int(torch.randint(H-h+1, (1,), generator=generator))
        x = int(torch.randint(W-w+1, (1,), generator=generator))
    elif mode == 'xai':
        # Integral-image exact exhaustive fixed-size window sum, O(HW), no approximate grid.
        sat = F.pad(heat.double(), (1,0,1,0)).cumsum(0).cumsum(1)
        sums = sat[h:,w:]-sat[:-h,w:]-sat[h:,:-w]+sat[:-h,:-w]
        index = int(sums.reshape(-1).argmax()); y, x = divmod(index, W-w+1)
    else: raise ValueError('Unknown crop location mode.')
    return y, x, h, w

def dispersion(heatmap):
    # Same 32x32 normalized coordinates for comparable entropy across patch grids/resolutions.
    heat = torch.as_tensor(heatmap).float().detach().cpu()
    if heat.ndim != 2 or not torch.isfinite(heat).all() or (heat < 0).any() or heat.sum() <= 0:
        raise ValueError('Undefined dispersion: map is nonfinite/negative/zero.')
    p = F.interpolate(heat[None,None], size=(32,32), mode='bilinear', align_corners=False)[0,0]
    p = p/p.sum(); nz = p[p>0]
    entropy = -(nz*nz.log()).sum()
    yy, xx = torch.meshgrid(torch.linspace(0,1,32), torch.linspace(0,1,32), indexing='ij')
    cy, cx = (p*yy).sum(), (p*xx).sum()
    return {'normalized_entropy': float(entropy/math.log(p.numel())),
            'effective_support': float(torch.exp(entropy)/p.numel()),
            'spatial_variance': float((p*((yy-cy)**2+(xx-cx)**2)).sum()),
            'centroid_y': float(cy), 'centroid_x': float(cx)}


In [ ]:
def normalize_xai_maps(maps,labels):
    if len(maps)!=len(labels): raise ValueError('XAI map/label count mismatch')
    valid=[];rejected=[];per={}
    for label,m in zip(labels,maps):
        m=torch.as_tensor(m).detach().float().cpu()
        if m.ndim!=2 or not torch.isfinite(m).all() or (m<0).any() or m.sum()<=0 or m.max()-m.min()<=1e-8:
            rejected.append({'concept':label,'reason':'nonfinite/zero/constant map'})
            continue
        small=F.interpolate((m/m.sum())[None,None],(32,32),mode='bilinear',align_corners=False)[0,0]
        if small.max()-small.min()<=1e-9:
            rejected.append({'concept':label,'reason':'constant after resizing'});continue
        small=small/small.sum()
        valid.append(small);per[label]=small
    if not valid: return None,per,rejected
    aggregate=torch.stack(valid).mean(0)
    return aggregate,per,rejected


def refresh_xai(xai,image,concepts,step,folder,role='adversarial',last_valid=None):
    """Writes per-concept and aggregate maps; XAI measurement failures cannot discard optimized PNGs."""
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    try:
        maps,meta=xai.maps(tensor_image(image),concepts)
        labels=meta.get('map_labels',concepts)
        if len(labels)!=len(maps): raise ValueError('Map count != labels count')
        aggregate,per,rejected=normalize_xai_maps(maps,labels)
        if aggregate is None: raise ValueError('No informative XAI map: '+str(rejected))
        valid=True;failure=None
    except Exception as e:
        # Deliberate CARA fail-safe; error is always logged and flag is false.
        valid=False;failure=f'{type(e).__name__}: {e}'
        meta={'error':failure};rejected=[];per={}
        aggregate=last_valid.detach().cpu() if last_valid is not None else None
    if aggregate is not None:
        H,W=image.shape[-2:]
        full=F.interpolate(aggregate[None,None],size=(H,W),mode='bilinear',align_corners=False)[0,0]
        arrays={'aggregate_32':aggregate.numpy()}
        for i,(label,m) in enumerate(per.items()): arrays[f'concept_{i:02d}']=m.numpy()
        np.savez_compressed(folder/f'{role}_{step:03d}.npz',**arrays)
    else: full=None
    record={'role':role,'step':step,'valid':valid,'fallback_used':bool(not valid and aggregate is not None),
            'failure':failure,'method':XAI_METHOD,'concepts':concepts,
            'concept_labels':list(per),'excluded_maps':rejected,
            'dispersion':dispersion(aggregate) if aggregate is not None else None,
            'metadata':meta}
    atomic_json(folder/f'{role}_{step:03d}.json',record)
    return full,record,aggregate


### CARA preprocessing — Grad-ECLIP XAI class

The functions `clip_encode_dense` and `grad_eclip` are the pinned upstream implementations above. This class only supplies preprocessing/model loading and reports one heatmap for each selected concept.

In [ ]:
class GradEclipXAI:
    def __init__(self, device):
        import clip
        self.clip = clip
        # Original OpenAI CLIP FP16 GPU model; separate from the HF ensemble.
        model, _ = clip.load('ViT-B/16', device=device, jit=False)
        model.eval().requires_grad_(False)
        self.model, self.device = model,torch.device(device)
        self.ns = dict(torch=torch,F=F,clipmodel=model,
            clip_inres=int(model.visual.input_resolution),clip_ksize=model.visual.conv1.kernel_size)
        exec(compile(AUTHOR_SOURCES['gradeclip_author_functions.py'],'<gradeclip-author>','exec'),self.ns)

    def maps(self, image, concepts):
        # Author imgprocess recipe: resize to nearest patch multiple while preserving aspect.
        image = reduced_image(image)
        w,h = image.size; ph,pw = self.ns['clip_ksize']
        nw,nh = max(pw,int(w/pw+.5)*pw),max(ph,int(h/ph+.5)*ph)
        preprocess = Compose([Resize((nh,nw), interpolation=InterpolationMode.BICUBIC),ToTensor(),
            Normalize((.48145466,.4578275,.40821073),(.26862954,.26130258,.27577711))])
        x = preprocess(image).unsqueeze(0).to(self.device).requires_grad_(True)
        with torch.no_grad():
            text = unit(self.model.encode_text(self.clip.tokenize(concepts,truncate=False).to(self.device)))
        with torch.enable_grad(), contextlib.redirect_stdout(io.StringIO()):
            outputs,_,vs,qs,ks,_,attention_outputs,grid = self.ns['clip_encode_dense'](x,n=1)
            scores = (unit(outputs[:,0])@text.T)[0]
            maps = [self.ns['grad_eclip'](s,qs,ks,vs,attention_outputs,grid).detach().float().cpu()
                    for s in scores]
        return maps,{'map_labels':list(concepts),'grid':list(grid),'model':'OpenAI ViT-B/16','n':1,
                     'author_forward_cosine':scores.detach().float().cpu().tolist()}


### CARA preprocessing — Qwen3-VL concept-token attention XAI class

This integrates unmodified SAGA helpers (`get_token_indices`, `build_attention_matrix`, `get_token_img_shape`) with **CARA-specific per-concept token grouping**. A single generation is reused for all concepts. This extension is not a claim of the SAGA author implementation.

In [ ]:
class ConceptMatchError(ValueError):
    def __init__(self,message,details):
        super().__init__(message);self.details=details

def match_finer_generated_tokens(tokenizer,token_ids,concepts,content_offsets):
    ids=[int(x) for x in token_ids]
    text=tokenizer.decode(ids,skip_special_tokens=False,clean_up_tokenization_spaces=False)
    # Prefix decoding preserves real BPE spans; do not match 'car' inside 'carpet'.
    prefixes=[tokenizer.decode(ids[:i],skip_special_tokens=False,clean_up_tokenization_spaces=False)
              for i in range(len(ids)+1)]
    if any(not text.startswith(p) for p in prefixes) or any(len(a)>len(b) for a,b in zip(prefixes,prefixes[1:])):
        raise ConceptMatchError('Non-monotonic generated-token decoding; cannot align concepts.',
                                {'generated_text':text,'concepts':concepts})
    spans=[(len(prefixes[i]),len(prefixes[i+1])) for i in range(len(ids))]
    allowed=set(content_offsets);selected=set();matches=[]
    # Approved rule: content words normalized by SAGA; no article/phrase exactness,
    # synonyms or stemming. Match complete decoded words before mapping back to BPEs.
    word_pattern=r"\w+(?:['’-]\w+)*"
    concept_words={phrase:{SAGA_NS['clean_token'](m.group()) for m in re.finditer(word_pattern,phrase)
                          if SAGA_NS['is_content_word'](m.group())} for phrase in concepts}
    for phrase in concepts:
        occurrences=[(m.start(),m.end()) for m in re.finditer(word_pattern,text)
                     if SAGA_NS['clean_token'](m.group()) in concept_words[phrase]]
        offsets=sorted({i for i,(a,b) in enumerate(spans) if i in allowed
                        and any(a<end and b>start for start,end in occurrences)})
        if offsets:
            selected.update(offsets);matches.append({'concept':phrase,'char_spans':occurrences,'token_offsets':offsets})
    details={'generated_text':text,'concepts':concepts,'matched_concepts':[r['concept'] for r in matches],
             'matches':matches,'selected_token_offsets':sorted(selected),
             'generated_token_ids':ids,'concept_content_words':{p:sorted(w) for p,w in concept_words.items()},
             'matching':'whole content words using SAGA normalization; no synonyms/lemmatization'}
    if not selected: raise ConceptMatchError('No generated content tokens match the FINER concepts.',details)
    return details


In [ ]:
class QwenAttentionXAI:
    def __init__(self,captioner):
        self.captioner=captioner

    @torch.no_grad()
    def maps(self,image,concepts):
        cap=self.captioner;inputs=cap.saga_batch(image);tokenizer=cap.processor.tokenizer
        outputs=cap.model.generate(**inputs,max_new_tokens=50,
            return_dict_in_generate=True,output_attentions=True,do_sample=False)
        decoded=cap.processor.batch_decode(outputs['sequences'],skip_special_tokens=False,
                                               clean_up_tokenization_spaces=False)[0]
        info=SAGA_NS['get_token_indices'](decoded,tokenizer,outputs,inputs,'qwen3_vl')
        ids=outputs.sequences[0][info['output_start']:info['output_end']]
        allowed=SAGA_NS['filter_generated_tokens'](tokenizer,ids,True)
        match=match_finer_generated_tokens(tokenizer,ids,concepts,allowed)
        matrix=SAGA_NS['build_attention_matrix'](outputs,layer_idx=QWEN_ATTENTION_LAYER)
        gh,gw=SAGA_NS['get_token_img_shape'](inputs,cap.processor,'qwen3_vl')
        start,end=info['vision_start'],info['vision_end']
        if (end-start)!=(gh*gw):raise RuntimeError('Vision grid/token mismatch')
        maps=[];labels=[]
        for matched in match['matches']:
            rows=[]
            for offset in matched['token_offsets']:
                row=info['output_start']+offset # SAGA original causal-row alignment
                if row>=matrix.shape[0]: continue
                value=matrix[row][start:end].float()
                if torch.isfinite(value).all() and value.sum()>0:
                    rows.append((value/value.sum()).reshape(gh,gw))
            if rows:
                maps.append(torch.stack(rows).mean(0).cpu())
                labels.append(matched['concept'])
        caption=tokenizer.decode(ids,skip_special_tokens=True)
        del outputs,matrix
        if not maps: raise ConceptMatchError('No matched concept with valid image attention',match)
        return maps,{'map_labels':labels,'layer':QWEN_ATTENTION_LAYER,'model':QWEN_ID,
                     'generated_caption':caption,'matching':match,
                     'aggregation':'SAGA generation-attention rows; CARA per-FINER-concept grouping'}


## Feature Extractors

The following three cells correspond to the explicitly requested CLIP backbones; each defines a separate class. `Qwen3VL` appears in its own cell. `ClipEnsemble` combines them with configurable image/text weights. Actual author CLIP implementation is accessed through pretrained `CLIPModel`, not misrepresented as a copy of MPCAttack source.

In [ ]:
class ClipBackbone:
    NAME = None
    def __init__(self,revisions):
        from transformers import CLIPModel, AutoTokenizer
        self.model_id=CLIP_IDS[self.NAME]
        self.device=torch.device(CLIP_DEVICE_MAP[self.NAME])
        self.model=CLIPModel.from_pretrained(self.model_id,revision=revisions[self.model_id],
                                             torch_dtype=torch.float32).eval().requires_grad_(False).to(self.device)
        self.tokenizer=AutoTokenizer.from_pretrained(self.model_id,revision=revisions[self.model_id])

    def encode_image(self,image):
        x=F.interpolate(image.to(self.device),size=(224,224),mode='bicubic',align_corners=False,
                        antialias=True).clamp(0,1)
        mean=x.new_tensor([.48145466,.4578275,.40821073])[None,:,None,None]
        std=x.new_tensor([.26862954,.26130258,.27577711])[None,:,None,None]
        return unit(self.model.get_image_features(pixel_values=(x-mean)/std))

    @torch.no_grad()
    def encode_text(self,texts):
        tokens=self.tokenizer(texts,padding=True,truncation=False,return_tensors='pt')
        if tokens.input_ids.shape[1]>self.model.config.text_config.max_position_embeddings:
            raise ValueError(f'{self.NAME}: caption exceeds CLIP context limit')
        return unit(self.model.get_text_features(**tokens.to(self.device)))


### Feature Extractor — CLIP ViT-g/14

In [ ]:
class CLIPG14(ClipBackbone):
    NAME = 'G14'


### Feature Extractor — CLIP ViT-B/16

In [ ]:
class CLIPB16(ClipBackbone):
    NAME = 'B16'


### Feature Extractor — CLIP ViT-B/32

In [ ]:
class CLIPB32(ClipBackbone):
    NAME = 'B32'


### Feature Extractor — Qwen3-VL-8B

Qwen generates caption banks, FINER object phrases and negative phrases; the same class supplies SAGA-compatible attention inputs. The Qwen adapter is CARA glue, not a Qwen training source file.

In [ ]:
# Instruction prompts copied verbatim from released MPC/SAGA code.
CAPTION_PROMPT = 'Describe this image in one concise sentence, no longer than 20 words.'
EVAL_CAPTION_PROMPT = CAPTION_PROMPT
ATTENTION_PROMPT = 'Describe this image, no longer than 25 words.'

class QwenCaptioner:
    def __init__(self, model_id, revision, device, attention=False):
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.device = torch.device(device)
        processor_options={} if QWEN_MAX_PIXELS is None else {'max_pixels':QWEN_MAX_PIXELS}
        self.processor=AutoProcessor.from_pretrained(model_id,revision=revision,**processor_options)
        with torch.cuda.device(self.device):
            dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        # Place the8B model after accounting for resident CLIPs; no quantization/substitution.
        max_memory={}
        for i in range(torch.cuda.device_count()):
            free,_=torch.cuda.mem_get_info(i)
            max_memory[i]=max(1,int(free-QWEN_GPU_RESERVE_GIB*2**30))
        self.model=AutoModelForImageTextToText.from_pretrained(model_id,revision=revision,
            torch_dtype=dtype,attn_implementation='eager' if attention else 'sdpa',
            device_map='auto',max_memory=max_memory,low_cpu_mem_usage=True)
        self.device=self.model.device
        self.model.eval().requires_grad_(False)

    def batch(self, messages, image=None):
        prompt = messages if isinstance(messages,str) else self.processor.apply_chat_template(
            messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
        kwargs = {'text': [prompt], 'return_tensors': 'pt'}
        if image is not None: kwargs['images'] = [image]
        return self.processor(**kwargs).to(self.device)

    @torch.inference_mode()
    def generate(self, messages, image=None, sample=False, max_new_tokens=96):
        batch = self.batch(messages,image)
        opts = dict(max_new_tokens=max_new_tokens, do_sample=sample, use_cache=True)
        if sample: opts.update(temperature=CAPTION_TEMPERATURE, top_p=CAPTION_TOP_P)
        ids = self.model.generate(**batch, **opts)
        result = self.processor.batch_decode(ids[:,batch['input_ids'].shape[1]:],
                      skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        if not result: raise ValueError('VLM returned an empty response.')
        return result

    def caption(self, image, sample=False, evaluation=False):
        messages = [{'role':'user','content':[{'type':'image'},
            {'type':'text','text': EVAL_CAPTION_PROMPT if evaluation else CAPTION_PROMPT}]}]
        return self.generate(messages,image,sample,CAPTION_MAX_NEW_TOKENS)

    def object_record(self,caption):
        # Original FINER512-token truncation, prompt builder and parser.
        prompt=FINER_NS['build_objects_prompt'](self.processor.tokenizer,caption)
        raw=self.generate(prompt,max_new_tokens=64)  # FINER build_generate_obj_parser default
        phrase=FINER_NS['parse_objects_output'](raw)
        objects=FINER_NS['_split_objects_phrase'](phrase)
        if not 1<=len(objects)<=5: raise ValueError(f'FINER produced no valid object list: {raw!r}')
        return {'raw_response':raw,'phrase':phrase,'objects':objects}

    def saga_batch(self,image):
        # Same image serialization/vision preparation as SAGA attention.py.
        import base64
        from qwen_vl_utils import process_vision_info
        image_processor=self.processor.image_processor
        if getattr(image_processor,'min_pixels',None) is None:image_processor.min_pixels=200704
        if getattr(image_processor,'max_pixels',None) is None:image_processor.max_pixels=1003520
        buffer=io.BytesIO();image.save(buffer,format='PNG')
        messages=[{'role':'user','content':[
            {'type':'image','image':'data:image/png;base64,'+base64.b64encode(buffer.getvalue()).decode()},
            {'type':'text','text':ATTENTION_PROMPT}]}]
        images,videos,_=process_vision_info(messages,image_patch_size=16,
            return_video_kwargs=True,return_video_metadata=True)
        text=self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        return self.processor(text=text,images=images,videos=videos,do_resize=False,
                              return_tensors='pt').to(self.device)


### Feature Extractors — CLIP ensemble and anchor banks

In [ ]:
def unit(x):
    x = x.float()
    if not torch.isfinite(x).all() or (x.norm(dim=-1) < 1e-12).any():
        raise ValueError('Nonfinite or zero feature vector.')
    return F.normalize(x, dim=-1)

def fuse_features(images, texts, image_weights, text_weights):
    if not len(images) == len(texts) == len(image_weights) == len(text_weights):
        raise ValueError('Backbone fusion lengths differ.')
    blocks = []
    for im, tx, wi, wt in zip(images, texts, image_weights, text_weights):
        if not math.isfinite(wi+wt) or min(wi, wt) < 0 or wi+wt <= 0:
            raise ValueError('Fusion weights must be finite, nonnegative, not both zero.')
        blocks.append(wi*unit(im) + wt*unit(tx))
    return torch.cat(blocks, dim=-1)

def nearest_mean_index(bank):
    if bank.ndim != 2 or len(bank) < 1: raise ValueError('Expected [K,D] bank.')
    return int(((bank-bank.mean(0, keepdim=True))**2).sum(-1).argmin())


In [ ]:
class ClipEnsemble:
    def __init__(self,revisions):
        self.models={'G14':CLIPG14(revisions),'B16':CLIPB16(revisions),'B32':CLIPB32(revisions)}
        self.device=torch.device(ATTACK_DEVICE)

    def images(self,image):
        return [self.models[name].encode_image(image).to(self.device) for name in BACKBONES]

    @torch.no_grad()
    def texts(self,captions):
        return [self.models[name].encode_text(captions).to(self.device) for name in BACKBONES]

    def query(self,image):
        return torch.cat(self.images(image),dim=-1)

    @torch.no_grad()
    def anchor_bank(self,image,captions):
        images=self.images(image);texts=self.texts(captions)
        return fuse_features(images,texts,[IMAGE_WEIGHTS[n] for n in BACKBONES],
                             [TEXT_WEIGHTS[n] for n in BACKBONES])

    @torch.no_grad()
    def anchors(self,source,target,banks):
        target_bank=self.anchor_bank(target,banks['target']['captions'])
        source_bank=self.anchor_bank(source,banks['source']['captions'])
        hard_negatives=self.anchor_bank(source,banks['source']['negative_captions'])
        j=nearest_mean_index(target_bank)
        # Keep text-only embeddings for visualization (unmodified text encoders).
        def joined(captions):
            return torch.cat(self.texts(captions),dim=-1).cpu().tolist()
        metadata={'target_index':j,'target_caption':banks['target']['captions'][j],
                  'target_distance_to_mean':((target_bank-target_bank.mean(0))**2).sum(-1).sqrt().cpu().tolist(),
                  'target_bank':target_bank.cpu().tolist(),'source_bank':source_bank.cpu().tolist(),
                  'negative_bank':hard_negatives.cpu().tolist(),
                  'text_target':joined(banks['target']['captions']),
                  'text_source':joined(banks['source']['captions']),
                  'text_negative':joined(banks['source']['negative_captions']),
                  'negative_captions':banks['source']['negative_captions']}
        return target_bank[j:j+1].detach(),hard_negatives.detach(),metadata


## Loss Function

MPCAttack upstream `info_nce_loss` is embedded unmodified above. CARA extends it from one source negative to **K=5 FINER hard-negative caption anchors**. The optimizer **maximizes** `-cross_entropy`, following MPC sign ascent.

In [ ]:
def infonce(query, positive, negatives, temperature, omega=2.0):
    if not 0 < temperature < float('inf'): raise ValueError('Invalid temperature.')
    if not 0 < omega < float('inf'): raise ValueError('Invalid positive omega.')
    q, p, n = unit(query), unit(positive), unit(negatives)
    if q.ndim != 2 or p.shape != q.shape or n.ndim != 2 or n.shape[1] != q.shape[1] or len(n) < 1:
        raise ValueError('Expected query/positive [B,D], negatives [K,D].')
    logits = torch.cat([omega*(q*p).sum(-1, keepdim=True), q@n.T], dim=1)/temperature
    return F.cross_entropy(logits, torch.zeros(len(q), dtype=torch.long, device=q.device))

def mpc_infonce_score(query,positive,negatives,temperature=.2,omega=2.0):
    # MPC returns negative CE and performs ascent. K=1 equals author info_nce_loss.
    return -infonce(query,positive,negatives,temperature,omega)

def initialize_mpc_delta(source_cpu,device):
    # Exact MPC expression, in the author's [0,255] optimization space.
    return (16/255*torch.randn_like(source_cpu).to(device)).requires_grad_(True)

def mpc_momentum_step(delta,gradient,momentum,alpha=1.,epsilon=16.,decay=.9):
    if not torch.isfinite(gradient).all(): raise FloatingPointError('Nonfinite MPC gradient.')
    momentum=decay*momentum+gradient
    return (delta.detach()+alpha*momentum.sign()).clamp(-epsilon,epsilon).requires_grad_(True),momentum.detach()


## Attack

`CARAAttack` follows a straightforward class-based optimization loop: anchor preparation → XAI measure → crop → CLIP query → InfoNCE → MI-FGSM-like update → projection → checkpoint. The original MPCAttack / SAGA functions are kept separate in original-source cells; this controller is the CARA hypothesis implementation.

In [ ]:
class CARAAttack:
    def __init__(self,ensemble,xai):
        self.ensemble=ensemble
        self.xai=xai

    def attack(self,source,target,banks,folder,seed):
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        seed_everything(seed)
        source255=(source.to(ATTACK_DEVICE)*255).round()
        delta=initialize_mpc_delta(source,ATTACK_DEVICE)
        momentum=torch.zeros_like(delta)
        positive,negatives,anchor_data=self.ensemble.anchors(source255/255,target.to(ATTACK_DEVICE),banks)
        assert negatives.shape[0]==K and K==len(banks['source']['negative_captions'])
        atomic_json(folder/'anchors.json',anchor_data)
        generator=torch.Generator(device='cpu').manual_seed(seed)
        xai_folder=folder/'xai'
        if SAVE_STEP_IMAGES:
            steps_folder=folder/'steps';steps_folder.mkdir(exist_ok=True)
            tensor_image(((source255+delta)/255).clamp(0,1)).save(steps_folder/'step_000.png')
        if XAI_ON_TARGET:
            _,target_rec,_=refresh_xai(self.xai,target,banks['target']['selected_concepts'],
                                        0,xai_folder,'target')
        _,src_rec,_=refresh_xai(self.xai,source,banks['source']['selected_concepts'],
                                 0,xai_folder,'source')
        initial=((source255+delta)/255).clamp(0,1)
        heat,record,last_valid=refresh_xai(self.xai,initial,banks['source']['selected_concepts'],
                                            0,xai_folder,'adversarial')
        if heat is None:
            # Source map is a valid concept map alternative at initialization.
            stored=xai_folder/'source_000.npz'
            if not stored.exists():
                raise RuntimeError('No valid initial XAI map for source or adversarial image')
            last_valid=torch.tensor(np.load(stored)['aggregate_32'])
            heat=F.interpolate(last_valid[None,None],size=source.shape[-2:],mode='bilinear',
                               align_corners=False)[0,0]
            record['fallback_used']=True;record['failure']='Initial adversarial map unavailable; using source map'
            atomic_json(xai_folder/'adversarial_000.json',record)
            np.savez_compressed(xai_folder/'adversarial_000.npz',aggregate_32=last_valid.numpy())
        history=[];xai_log=[record];cached_box=None
        for step in range(1,STEPS+1):
            area=crop_area(step,STEPS,REFRESH_EVERY)
            mode=crop_mode(step,RANDOM_FROM_STEP)
            if mode=='xai':
                if cached_box is None:cached_box=select_box(heat,area,'xai',generator)
                box=cached_box
            else:box=select_box(heat,area,'random',generator)
            y,x,h,w=box
            context=torch.autograd.graph.save_on_cpu(pin_memory=True) if SAVED_TENSORS_ON_CPU else contextlib.nullcontext()
            with context:
                query=self.ensemble.query(((source255+delta)/255)[:,:,y:y+h,x:x+w])
                score=mpc_infonce_score(query,positive,negatives,TEMPERATURE,POSITIVE_OMEGA)
                if not torch.isfinite(score):raise FloatingPointError(f'NaN loss at step {step}')
                gradient,=torch.autograd.grad(score,delta)
            if not torch.isfinite(gradient).all() or float(gradient.abs().sum())==0:
                raise RuntimeError(f'Invalid/zero gradient at step {step}')
            new_delta,momentum=mpc_momentum_step(delta,gradient,momentum,ALPHA_255,
                                                 EPSILON_255,MOMENTUM_DECAY)
            if MASK_UPDATES_TO_CROP:
                # Strict region update is CARA-specific, NOT the original MPC MI-FGSM update.
                mask=torch.zeros_like(delta);mask[:,:,y:y+h,x:x+w]=1
                new_delta=torch.where(mask.bool(),new_delta,delta.detach()).requires_grad_(True)
            delta=new_delta
            # Projection onto intersection of epsilon-ball and valid RGB range.
            delta=torch.maximum(torch.minimum(delta.detach(),255-source255),-source255)
            delta=delta.clamp(-EPSILON_255,EPSILON_255).requires_grad_(True)
            with torch.no_grad():
                p=unit(positive);n=unit(negatives);q=unit(query)
                target_sim=float((q*p).sum()); source_sim=float((q@n.T).max())
            history.append({'step':step,'loss_before_update':float(-score.detach()),
                            'score_before_update':float(score.detach()),
                            'target_similarity':target_sim,'max_negative_similarity':source_sim,
                            'crop_mode':mode,'box_yxhw':list(box),'scheduled_area':area,
                            'actual_area':h*w/(INPUT_RES*INPUT_RES),
                            'gradient_l1':float(gradient.abs().sum())})
            del query,score,gradient
            if step % REFRESH_EVERY==0:
                if SAVE_STEP_IMAGES:
                    tensor_image(((source255+delta)/255).clamp(0,1)).save(
                        steps_folder/f'step_{step:03d}.png')
                cur=((source255+delta)/255).clamp(0,1)
                # XAI diagnostics always recorded; do not discard good 300-step result.
                new_heat,rec,new_valid=refresh_xai(self.xai,cur,banks['source']['selected_concepts'],
                                step,xai_folder,'adversarial',last_valid=last_valid)
                if new_heat is not None:heat=new_heat
                if new_valid is not None:last_valid=new_valid
                xai_log.append(rec);cached_box=None
                atomic_json(folder/'progress.json',{'completed_updates':step,
                                                     'history':history,'xai':xai_log})
                if step % LOG_EVERY==0:
                    print(f'  step {step:03d}/{STEPS}; {mode}; CE={history[-1]["loss_before_update"]:.4f}; '
                          f'XAI_valid={rec["valid"]}')
        adversarial=((source255+delta)/255).clamp(0,1).detach()
        with torch.no_grad():
            src_q=unit(self.ensemble.query(source255/255))
            adv_q=unit(self.ensemble.query(adversarial))
            target_q=unit(positive);negative_q=unit(negatives)
            metrics={'clean_global_infonce':float(infonce(src_q,positive,negatives,TEMPERATURE,POSITIVE_OMEGA)),
                     'adversarial_global_infonce':float(infonce(adv_q,positive,negatives,TEMPERATURE,POSITIVE_OMEGA)),
                     'clean_target_cosine':float((src_q*target_q).sum()),
                     'adversarial_target_cosine':float((adv_q*target_q).sum()),
                     'clean_negative_max_cosine':float((src_q@negative_q.T).max()),
                     'adversarial_negative_max_cosine':float((adv_q@negative_q.T).max())}
        return adversarial.cpu(),{'history':history,'xai':xai_log,'surrogate_proxies':metrics,
             'optimizer':'CARA multi-negative MPC InfoNCE, momentum sign ascent',
             'strict_region_mask':MASK_UPDATES_TO_CROP,'negative_anchor_count':len(negatives)}


### Attack — dataset loop, caches and exports

Per-sample errors are logged without silently excluding them from the requested count. A completed image is not deleted because a diagnostic XAI refresh failed. Output snapshots are saved at steps 0, 30, ..., 300.

In [ ]:
def caption_bank(captioner, image):
    captions, attempts = [], []
    for _ in range(MAX_BANK_ATTEMPTS):
        raw = captioner.caption(image, sample=True)
        cleaned = ' '.join(raw.split()); key = cleaned.casefold().rstrip('. ')
        duplicate = key in {c.casefold().rstrip('. ') for c in captions}
        attempts.append({'raw_response':raw,'accepted':not duplicate})
        if not duplicate: captions.append(cleaned)
        if len(captions)==K: break
    if len(captions)!=K:
        raise RuntimeError(f'Only {len(captions)}/{K} unique captions; increase attempt budget. No duplicate padding.')
    objects = [captioner.object_record(c) for c in captions]
    selected=[];seen=set()
    for row in objects:
        for phrase in row['objects']:
            key=phrase.casefold()
            if key not in seen: selected.append(phrase);seen.add(key)
    if not selected: raise ValueError('FINER bank has no objects.')
    return {'captions':captions,'caption_attempts':attempts,'object_records':objects,
            'selected_concepts':selected}
def bank_for_pair(pair,captioner,run_dir,identity):
    folder=run_dir/'banks';folder.mkdir(exist_ok=True)
    path=folder/f'{pair["index"]:05d}.json'
    bank_identity={'pair_index':pair['index'],'inputs':pair['input_sha256'],'run':digest(identity)}
    if path.exists():
        record=json.loads(path.read_text(encoding='utf-8'))
        payload={k:v for k,v in record.items() if k!='record_sha256'}
        if record['identity']!=bank_identity or digest(payload)!=record.get('record_sha256'):
            raise RuntimeError(f'Caption bank identity/hash mismatch: {path}')
        if any(len(record[r]['captions'])!=K for r in ('source','target')) or len(record['source'].get('negative_captions',[]))!=K:
            raise ValueError('Cached bank lacks K=5 source, target and hard-negative captions.')
        return record
    record={'identity':bank_identity}
    for j,role in enumerate(('source','target')):
        seed_everything(SEED+pair['index']*1000+j)
        record[role]=caption_bank(captioner,load_attack_image(pair[role]))
    # Generate exactly K distinct hard negatives from original FINER negative-object prompts.
    record['source'].update(negative_caption_bank(captioner,record['source'],record['target']['captions']))
    record['record_sha256']=digest(record)
    atomic_json(path,record)
    return record

def sample_complete(folder,expected):
    path=folder/'complete.json'
    if not path.exists(): return False
    record=json.loads(path.read_text(encoding='utf-8'))
    if record['identity']!=expected: raise RuntimeError(f'Completed sample identity mismatch: {folder}')
    for filename,sha in record['files'].items():
        p=folder/filename
        if not p.is_file() or file_hash(p)!=sha: raise RuntimeError(f'Artifact corrupted/missing: {p}')
    return True

def export_sample(folder,pair,source,target,adv,record,identity):
    delta=(adv-source).abs()
    if not torch.isfinite(adv).all() or float(delta.max())>EPSILON_255/255+1e-6:
        raise AssertionError('Adversarial tensor exceeds pixel constraint.')
    for role,x in [('source',source),('target',target),('adversarial',adv)]:
        tensor_image(x).save(folder/f'{role}.png')
    saved_source=np.asarray(Image.open(folder/'source.png')).astype(np.int16)
    saved_adv=np.asarray(Image.open(folder/'adversarial.png')).astype(np.int16)
    saved_linf=int(np.abs(saved_adv-saved_source).max())
    if saved_linf>EPSILON_255: raise AssertionError('Saved PNG violates epsilon.')
    record.update(index=pair['index'],inputs=pair,identity=identity,
        tensor_linf_255=float(delta.max()*255),saved_linf_255=saved_linf,
        source_size=list(source.shape[-2:]),target_size=list(target.shape[-2:]))
    atomic_json(folder/'attack.json',record)
    # Complete marker written last; partial samples are restarted from their deterministic seed.
    files={str(p.relative_to(folder)).replace('\\','/'):file_hash(p)
           for p in folder.rglob('*') if p.is_file() and p.name!='complete.json'}
    atomic_json(folder/'complete.json',{'identity':identity,'files':files})

def export_manifest(run_dir,pairs):
    rows=[]
    for pair in pairs:
        folder=run_dir/'samples'/f'{pair["index"]:05d}'
        if not (folder/'complete.json').exists(): continue
        row={'index':pair['index'],'sample_id':f'{pair["index"]:05d}',
            'images':{r:f'samples/{pair["index"]:05d}/{r}.png' for r in ('source','target','adversarial')},
            'original_paths':{r:pair[r] for r in ('source','target')},
            'caption_bank':f'banks/{pair["index"]:05d}.json'}
        rows.append(row)
    (run_dir/f'manifest_{RUN_MODE}_{SAMPLE_START}.jsonl').write_text(
        ''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
    return rows


In [ ]:
def stable_code_digest(functions):
    # Code-object repr contains process addresses. Describe nested constants recursively.
    import types, inspect
    def encode(value):
        if isinstance(value, types.CodeType):
            return {'bytecode':value.co_code.hex(),'names':list(value.co_names),
                    'varnames':list(value.co_varnames),'freevars':list(value.co_freevars),
                    'argcount':value.co_argcount,'kwonly':value.co_kwonlyargcount,
                    'constants':[encode(c) for c in value.co_consts]}
        if isinstance(value,(tuple,list)): return [encode(v) for v in value]
        if isinstance(value,frozenset): return sorted([encode(v) for v in value],key=repr)
        if isinstance(value,bytes): return {'bytes':value.hex()}
        return repr(value)
    return digest({f.__qualname__:encode(inspect.unwrap(f).__code__) for f in functions})


def run_identity(revisions):
    import types
    functions=[]
    namespace=globals()
    for value in list(namespace.values()):
        if isinstance(value,types.FunctionType) and value.__globals__ is namespace:
            functions.append(value)
        elif isinstance(value,type) and value.__module__==__name__:
            functions.extend(m for m in vars(value).values() if isinstance(m,types.FunctionType))
    config={'revision':'v6.2','note':RUN_NOTE,'method':XAI_METHOD,'seed':SEED,
            'K':K,'steps':STEPS,'refresh':REFRESH_EVERY,'random_from':RANDOM_FROM_STEP,
            'epsilon':EPSILON_255,'alpha':ALPHA_255,'tau':TEMPERATURE,'omega':POSITIVE_OMEGA,
            'momentum':MOMENTUM_DECAY,'strict_crop_mask':MASK_UPDATES_TO_CROP,
            'negative_attempts':NEGATIVE_ATTEMPTS,
            'caption_sampling':{'temperature':CAPTION_TEMPERATURE,'top_p':CAPTION_TOP_P,
                'max_new_tokens':CAPTION_MAX_NEW_TOKENS,'max_attempts':MAX_BANK_ATTEMPTS},
            'xai_config':{'layer':QWEN_ATTENTION_LAYER,'max_pixels':QWEN_MAX_PIXELS,
                'max_side':XAI_MAX_SIDE,'failure_policy':XAI_FAILURE_POLICY,'on_target':XAI_ON_TARGET},
            'runtime':{p:package_metadata.version(p) for p in ('torch','torchvision','transformers','numpy','Pillow')},'clip_ids':CLIP_IDS,
            'image_weights':IMAGE_WEIGHTS,'text_weights':TEXT_WEIGHTS,
            'qwen_id':QWEN_ID,'model_revisions':revisions,'author_sha256':AUTHOR_SHA256,
            'integration_code_sha256':stable_code_digest(functions)}
    return config


def execute_experiment(pairs):
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    revisions=resolve_revisions();identity=run_identity(revisions)
    run_dir=OUTPUT_ROOT/(RUN_NOTE+'_'+XAI_METHOD+'_'+digest(identity)[:12])
    run_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(run_dir/'config.json',identity)
    bank_results={};failures=[];captioner=None;ensemble=None;xai=None
    # Stage 1: Qwen caption banks, FINER objects, FINER hard negatives.
    try:
        for pair in pairs:
            try:
                if captioner is None:
                    captioner=QwenCaptioner(QWEN_ID,revisions[QWEN_ID],QWEN_DEVICE)
                bank_results[pair['index']]=bank_for_pair(pair,captioner,run_dir,identity)
            except Exception as exc:
                failure={'index':pair['index'],'stage':'bank','error':f'{type(exc).__name__}: {exc}'}
                failures.append(failure);atomic_json(run_dir/'failures.json',failures)
                if ABORT_ON_SAMPLE_ERROR:raise
    finally:
        del captioner;captioner=None;free_cuda()
    # Stage 2: CLIP ensemble, Grad-ECLIP or Qwen XAI, targeted attack.
    try:
        if bank_results:
            ensemble=ClipEnsemble(revisions)
            if XAI_METHOD=='grad_eclip':xai=GradEclipXAI(ATTACK_DEVICE)
            else:
                captioner=QwenCaptioner(QWEN_ID,revisions[QWEN_ID],QWEN_DEVICE,attention=True)
                xai=QwenAttentionXAI(captioner)
        for pair in pairs:
            if pair['index'] not in bank_results:continue
            folder=run_dir/'samples'/f'{pair["index"]:05d}'
            folder.mkdir(parents=True,exist_ok=True)
            expected={'run':digest(identity),'inputs':pair['input_sha256'],
                      'bank':bank_results[pair['index']]['record_sha256'],'index':pair['index']}
            try:
                if sample_complete(folder,expected):
                    print('SKIP completed:',pair['index']);continue
                source=image_tensor(load_attack_image(pair['source']))
                target=image_tensor(load_attack_image(pair['target']))
                adversarial,record=CARAAttack(ensemble,xai).attack(
                     source,target,bank_results[pair['index']],folder,SEED+pair['index'])
                export_sample(folder,pair,source,target,adversarial,record,expected)
                export_manifest(run_dir,pairs)
                del source,target,adversarial;free_cuda()
            except Exception as exc:
                failure={'index':pair['index'],'stage':'attack','error':f'{type(exc).__name__}: {exc}'}
                failures.append(failure);atomic_json(run_dir/'failures.json',failures)
                print('FAILED sample',pair['index'],failure['error'])
                if ABORT_ON_SAMPLE_ERROR:raise
    finally:
        del ensemble,xai,captioner;free_cuda()
    rows=export_manifest(run_dir,pairs)
    if not rows:
        print('No completed samples. See failures.json.');return run_dir,rows,[]
    evaluation=[]
    if EVALUATE_CAPTIONS:
        try:
            _,evaluation=evaluate_caption_outputs(run_dir,rows,revisions)
        except Exception as exc:
            failures.append({'stage':'evaluation','error':f'{type(exc).__name__}: {exc}'})
            atomic_json(run_dir/'failures.json',failures)
            if ABORT_ON_SAMPLE_ERROR:raise
    review=human_review_table(run_dir,rows,evaluation)
    atomic_json(review.with_suffix('.asr.json'),report_human_asr(review))
    atomic_json(run_dir/'pilot_summary.json',{'requested':len(pairs),
        'completed':len(rows),'failed':failures,'evaluation_rows':len(evaluation)})
    print(f'Completed {len(rows)}/{len(pairs)}; failures={len(failures)}. Results: {run_dir}')
    return run_dir,rows,evaluation


## Evaluation

Qwen2.5-VL captioning runs *after* attack; it is not the surrogate. CLIP text-cosine and InfoNCE values are diagnostics, **not targeted ASR**. Manual review CSV stores `target_success` and `clean_already_target` fields.

In [ ]:
def evaluate_caption_outputs(run_dir,rows,revisions):
    eval_config={'model':EVAL_MODEL_ID,'revision':revisions[EVAL_MODEL_ID],
        'prompt':EVAL_CAPTION_PROMPT,'max_new_tokens':CAPTION_MAX_NEW_TOKENS,
        'do_sample':False,'max_pixels':QWEN_MAX_PIXELS,'transformers':package_metadata.version('transformers')}
    eval_dir=run_dir/'evaluation'/digest(eval_config)[:16];eval_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(eval_dir/'protocol.json',eval_config)
    evaluator=None;outputs=[]
    try:
        for row in rows:
            path=eval_dir/f'{row["index"]:05d}.json'
            inputs={r:file_hash(run_dir/row['images'][r]) for r in ('source','adversarial')}
            expected={'inputs':inputs,'protocol':digest(eval_config)}
            if path.exists():
                record=json.loads(path.read_text(encoding='utf-8'))
                if record['identity']!=expected: raise RuntimeError('Evaluation cache identity mismatch.')
                payload={k:v for k,v in record.items() if k!='record_sha256'}
                if record.get('record_sha256')!=digest(payload): raise RuntimeError('Evaluation response cache corrupted.')
            else:
                if evaluator is None: evaluator=QwenCaptioner(EVAL_MODEL_ID,revisions[EVAL_MODEL_ID],EVAL_DEVICE)
                seed_everything(SEED+row['index'])
                captions={r:evaluator.caption(open_rgb(run_dir/row['images'][r]),evaluation=True)
                          for r in ('source','adversarial')}
                record={'identity':expected,'index':row['index'],'captions':captions}
                record['record_sha256']=digest(record);atomic_json(path,record)
            outputs.append(record)
    finally:
        del evaluator;free_cuda()
    # Staged lifecycle: evaluate VLM first, release it, then load one CLIP at a time for text proxies.
    from transformers import CLIPModel,AutoTokenizer
    scores={r['index']:{} for r in rows}
    lookup={o['index']:o for o in outputs}
    for name in BACKBONES:
        ident=CLIP_IDS[name]
        model=CLIPModel.from_pretrained(ident,revision=revisions[ident],torch_dtype=torch.float32)
        model.eval().requires_grad_(False).to(EVAL_DEVICE)
        tokenizer=AutoTokenizer.from_pretrained(ident,revision=revisions[ident])
        with torch.no_grad():
            for row in rows:
                bank=json.loads((run_dir/row['caption_bank']).read_text(encoding='utf-8'))
                caps=lookup[row['index']]['captions']
                texts=[caps['source'],caps['adversarial']]+bank['target']['captions']+bank['source']['captions']
                tokens=tokenizer(texts,padding=True,return_tensors='pt',truncation=False).to(EVAL_DEVICE)
                if tokens.input_ids.shape[1]>model.config.text_config.max_position_embeddings:
                    raise ValueError('Evaluation caption exceeds CLIP token budget; inspect raw caption.')
                emb=unit(model.get_text_features(**tokens))
                tgt=unit(emb[2:2+K].mean(0,keepdim=True));src=unit(emb[2+K:].mean(0,keepdim=True))
                similarities=emb[:2]@torch.cat([tgt,src]).T
                scores[row['index']][name]={'clean_target':float(similarities[0,0]),
                    'adv_target':float(similarities[1,0]),'clean_source':float(similarities[0,1]),
                    'adv_source':float(similarities[1,1])}
        del model,tokenizer;free_cuda()
    results=[]
    for row in rows:
        idx=row['index'];record={'index':idx,'evaluation_protocol':digest(eval_config),**lookup[idx]['captions']}
        for key in ('clean_target','adv_target','clean_source','adv_source'):
            record[key+'_text_cosine_proxy']=float(np.mean([scores[idx][m][key] for m in BACKBONES]))
        record['target_text_cosine_gain_proxy']=record['adv_target_text_cosine_proxy']-record['clean_target_text_cosine_proxy']
        results.append(record)
    pd.DataFrame(results).to_csv(eval_dir/f'caption_proxies_{RUN_MODE}_{SAMPLE_START}.csv',index=False)
    atomic_json(eval_dir/f'raw_scores_{RUN_MODE}_{SAMPLE_START}.json',scores)
    return eval_dir,results

def human_review_table(run_dir,rows,caption_records):
    protocols={r['evaluation_protocol'] for r in caption_records}
    if len(protocols)>1: raise ValueError('Mixed evaluation protocols.')
    protocol=next(iter(protocols)) if protocols else 'image_only_no_vlm'
    path=run_dir/f'human_review_{RUN_MODE}_{SAMPLE_START}_{protocol[:16]}.csv'
    if path.exists():
        frame=pd.read_csv(path)
        if frame['index'].tolist()!=[r['index'] for r in rows]: raise ValueError('Review table indices mismatch.')
        return path
    lookup={r['index']:r for r in caption_records}
    records=[]
    for row in rows:
        bank=json.loads((run_dir/row['caption_bank']).read_text(encoding='utf-8'))
        record={'index':row['index'],'evaluation_protocol':protocol,**{r:str(run_dir/p) for r,p in row['images'].items()},
            'source_bank':json.dumps(bank['source']['captions'],ensure_ascii=False),
            'target_bank':json.dumps(bank['target']['captions'],ensure_ascii=False),
            'clean_vlm_caption':lookup.get(row['index'],{}).get('source',''),
            'adv_vlm_caption':lookup.get(row['index'],{}).get('adversarial',''),
            'target_success':'','clean_already_target':'','review_notes':''}
        records.append(record)
    pd.DataFrame(records).to_csv(path,index=False)
    return path

def report_human_asr(review_csv):
    frame=pd.read_csv(review_csv)
    if 'evaluation_protocol' in frame and (frame['evaluation_protocol']=='image_only_no_vlm').any():
        return {'status':'no_vlm_caption_evaluation','n':len(frame)}
    for col in ('target_success','clean_already_target'):
        if frame[col].isna().any():
            return {'status':'pending_human_labels','n':len(frame),'labeled':int(frame['target_success'].notna().sum())}
        if not frame[col].isin([0,1]).all(): raise ValueError(f'{col} must contain only 0/1.')
    eligible=frame[frame['clean_already_target']==0]
    return {'status':'human_labeled','n':len(frame),'targeted_asr_all':float(frame['target_success'].mean()),
        'n_clean_not_target':len(eligible),
        'targeted_asr_clean_not_target':float(eligible['target_success'].mean()) if len(eligible) else None}


## Visualization

Run visualization cells after the attack. They load saved artifacts, do not re-run Qwen3-VL or CLIP. All routines accept `run_dir` and `sample_id` (numeric index). Per-step heatmaps may be reused from the last valid map when Qwen concept matching fails; check `fallback_used` in JSON.

In [ ]:
def sample_folder(run_dir,sample_id):
    folder=Path(run_dir)/'samples'/f'{int(sample_id):05d}'
    if not folder.exists():raise FileNotFoundError(folder)
    return folder


def load_xai_array(folder,role='adversarial',step=0,key='aggregate_32'):
    path=Path(folder)/'xai'/f'{role}_{step:03d}.npz'
    if not path.exists():return None
    with np.load(path) as data:
        if key not in data:return None
        return data[key].copy()


def overlay_attention(image,map_array,ax,caption=None):
    ax.imshow(image)
    if map_array is not None:
        ax.imshow(map_array,cmap='inferno',alpha=.48,extent=(0,image.width,image.height,0))
    ax.axis('off')
    if caption:ax.set_title(caption)


### Visualization — adversarial images across steps

In [ ]:
def visualize_steps(run_dir,sample_id,steps=(0,30,60,90,120,150,180,210,240,270,300)):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    fig,axes=plt.subplots(3,4,figsize=(17,11));axes=axes.ravel()
    for ax,step in zip(axes,steps):
        path=folder/'steps'/f'step_{step:03d}.png'
        if path.exists():ax.imshow(open_rgb(path));ax.set_title(f'Update {step}')
        else:ax.set_title(f'Missing step {step}')
        ax.axis('off')
    for ax in axes[len(steps):]:ax.axis('off')
    fig.suptitle(f'Adversarial progression — sample {sample_id}')
    plt.tight_layout();plt.show()


### Visualization — source and target attention maps

In [ ]:
def visualize_source_target_attention(run_dir,sample_id):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    fig,axes=plt.subplots(2,3,figsize=(15,9))
    for row,role in enumerate(('source','target')):
        img=open_rgb(folder/f'{role}.png')
        heat=load_xai_array(folder,role,0)
        overlay_attention(img,None,axes[row,0],f'{role} original')
        overlay_attention(img,heat,axes[row,1],f'{role} XAI overlay')
        if heat is not None:axes[row,2].imshow(heat,cmap='inferno')
        axes[row,2].set_title(f'{role} heatmap');axes[row,2].axis('off')
    plt.tight_layout();plt.show()


### Visualization — caption embedding projection into 2D (PCA)

In [ ]:
def visualize_caption_space(run_dir,sample_id):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    anc=json.loads((folder/'anchors.json').read_text())
    groups=[('Source',np.asarray(anc['text_source'])),
            ('Target',np.asarray(anc['text_target'])),
            ('FINER-negative',np.asarray(anc['text_negative']))]
    all_values=np.concatenate([g[1] for g in groups],axis=0)
    center=all_values-all_values.mean(axis=0,keepdims=True)
    _,_,vt=np.linalg.svd(center,full_matrices=False)
    coords=center@vt[:2].T
    fig,ax=plt.subplots(figsize=(9,7)); start=0
    for label,values in groups:
        xy=coords[start:start+len(values)]
        ax.scatter(xy[:,0],xy[:,1],label=label,s=85)
        for i,(x,y) in enumerate(xy):ax.annotate(str(i+1),(x,y),xytext=(4,4),textcoords='offset points')
        start+=len(values)
    ax.set(xlabel='PC1',ylabel='PC2',title='Text-only CLIP concatenated embeddings (PCA)')
    ax.legend();ax.grid(alpha=.2);plt.show()


### Visualization — XAI heatmaps and dispersion every 30 steps

In [ ]:
def visualize_attention_timeline(run_dir,sample_id):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    steps=list(range(0,STEPS+1,REFRESH_EVERY))
    fig,axes=plt.subplots(3,4,figsize=(16,11));axes=axes.ravel()
    original=open_rgb(folder/'source.png')
    for ax,step in zip(axes,steps):
        heat=load_xai_array(folder,'adversarial',step)
        step_file=folder/'steps'/f'step_{step:03d}.png'
        image=open_rgb(step_file) if step_file.exists() else original
        overlay_attention(image,heat,ax,f'Adv XAI step {step}')
        status_file=folder/'xai'/f'adversarial_{step:03d}.json'
        if status_file.exists():
            rec=json.loads(status_file.read_text())
            if rec.get('fallback_used'):ax.set_title(f'Step {step} (fallback)')
    for ax in axes[len(steps):]:ax.axis('off')
    plt.tight_layout();plt.show()


def visualize_xai_dispersion(run_dir,sample_id):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    records=json.loads((folder/'attack.json').read_text())['xai']
    x=[r['step'] for r in records]
    y=[r['dispersion']['normalized_entropy'] if r['dispersion'] else float('nan') for r in records]
    plt.figure(figsize=(10,4));plt.plot(x,y,'o-');plt.axvline(RANDOM_FROM_STEP,linestyle='--')
    plt.xlabel('Completed updates');plt.ylabel('Normalized attention entropy')
    plt.title('XAI dispersion through optimization');plt.show()


### Visualization — InfoNCE loss and target/negative similarities

In [ ]:
def visualize_loss_similarity(run_dir,sample_id):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    history=json.loads((folder/'attack.json').read_text())['history']
    steps=[v['step'] for v in history]
    fig,axes=plt.subplots(1,2,figsize=(15,5))
    axes[0].plot(steps,[v['loss_before_update'] for v in history],label='InfoNCE CE')
    axes[0].set(xlabel='Update',ylabel='CE (lower is better)',title='Targeted contrastive objective')
    axes[1].plot(steps,[v['target_similarity'] for v in history],label='Target cosine')
    axes[1].plot(steps,[v['max_negative_similarity'] for v in history],label='Max negative cosine')
    axes[1].set(xlabel='Update',ylabel='Cosine',title='Similarity before each update')
    for ax in axes:ax.axvline(RANDOM_FROM_STEP,linestyle='--',color='gray');ax.grid(alpha=.2);ax.legend()
    plt.tight_layout();plt.show()


### Visualization — per-concept maps at an XAI refresh

In [ ]:
def visualize_concept_maps(run_dir,sample_id,role='adversarial',step=0):
    import matplotlib.pyplot as plt
    folder=sample_folder(run_dir,sample_id)
    path=folder/'xai'/f'{role}_{step:03d}.npz'
    if not path.exists():print('No saved XAI map at',path);return
    meta=json.loads(path.with_suffix('.json').read_text())
    with np.load(path) as data:
        names=list(data.keys())
        if not names:return
        fig,axes=plt.subplots(1,len(names),figsize=(4*len(names),4),squeeze=False)
        for idx,key in enumerate(names):
            label='Aggregate' if key=='aggregate_32' else (
                meta['concept_labels'][int(key.split('_')[-1])] if int(key.split('_')[-1])<len(meta['concept_labels']) else key)
            ax=axes[0,idx];ax.imshow(data[key],cmap='inferno');ax.set_title(label);ax.axis('off')
        plt.tight_layout();plt.show()


## Validation — deterministic CPU-friendly checks before GPU pilot

In [ ]:
def self_checks():
    import torch
    torch.manual_seed(17)
    q=torch.randn(1,9,requires_grad=True);p=torch.randn(1,9);n=torch.randn(1,9)
    torch.testing.assert_close(mpc_infonce_score(q,p,n,.2,2.),
                               MPC_NS['info_nce_loss'](q,p,n,temperature=.2,omega=2.))
    multi=torch.randn(5,9)
    score=mpc_infonce_score(q,p,multi,.2,2.)
    grad,=torch.autograd.grad(score,q)
    assert torch.isfinite(grad).all() and grad.abs().sum()>0
    assert crop_area(1)==.5 and crop_area(150)==13/18 and crop_area(300)==1
    assert crop_mode(149)=='xai' and crop_mode(150)=='random'
    assert validate_negative_phrase('a dog and a tree','a cat and a tree',1)
    assert not validate_negative_phrase('a dog and a tree','a cat and a bush',1)
    assert replace_object_in_caption('A brown dog runs.','a dog','a panda')=='A brown panda runs.'
    aggregate,_,discard=normalize_xai_maps([torch.zeros(8,8)],['invalid'])
    assert aggregate is None and discard
    area=.5; heat=torch.rand(16,16);assert select_box(heat,area,'xai',torch.Generator())[2]>0
    assert sample_count()==NUM_SAMPLES
    return {'K1_equals_MPC':True,'K5_gradient':True,'crop_schedule':True,'negative_replacement':True,
            'degenerate_heatmap_rejection':True}
SELF_CHECK_REPORT=self_checks()
print('SELF CHECKS:',SELF_CHECK_REPORT)


## Run — preview and pilot

The following cells actually run pretrained models on Kaggle GPUs; default `NUM_SAMPLES=5`, `XAI_METHOD="grad_eclip"`. For 200 samples, edit Config only after all 5 pilots generate valid artifacts and human evaluation is completed.

In [ ]:
PAIRS=[];RUN_DIR=None;MANIFEST_ROWS=[];CAPTION_EVALUATION=[]
if RUN_EXPERIMENT:
    validate_config()
    PAIRS=prepare_pairs()
    from IPython.display import display
    display(pd.DataFrame(PAIRS)[['index','source','target']])
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(len(PAIRS),2,figsize=(9,3*len(PAIRS)),squeeze=False)
    for pair,axis_row in zip(PAIRS,axes):
        for role,ax in zip(('source','target'),axis_row):
            ax.imshow(load_attack_image(pair[role]));ax.set_title(f"{pair['index']} {role}");ax.axis('off')
    plt.tight_layout();plt.show()


In [ ]:
if RUN_EXPERIMENT:
    RUN_DIR,MANIFEST_ROWS,CAPTION_EVALUATION=execute_experiment(PAIRS)


## Pilot verification — validate every saved sample before scaling

In [ ]:
def validate_pilot_results(run_dir,rows,requested):
    checks=[]
    for row in rows:
        idx=row['index'];folder=sample_folder(run_dir,idx)
        bank=json.loads((Path(run_dir)/row['caption_bank']).read_text(encoding='utf-8'))
        attack=json.loads((folder/'attack.json').read_text(encoding='utf-8'))
        error=[]
        if len(bank['source']['captions'])!=K or len(bank['target']['captions'])!=K:
            error.append('Caption bank K mismatch')
        if len(bank['source'].get('negative_captions',[]))!=K:
            error.append('Negative captions K mismatch')
        if any(not validate_negative_phrase(r['source_phrase'],r['negative_phrase'],r['replace_index'])
               for r in bank['source'].get('negative_records',[])):
            error.append('Invalid FINER object replacement')
        if len(attack['history'])!=STEPS or len(attack['xai'])!=1+STEPS//REFRESH_EVERY:
            error.append('Missing optimization/XAI steps')
        if attack['saved_linf_255']>EPSILON_255:error.append('Exceeds L∞ epsilon')
        if len(list((folder/'steps').glob('step_*.png')))!=1+STEPS//REFRESH_EVERY:
            error.append('Missing step image checkpoints')
        xai_failed=sum(not record['valid'] for record in attack['xai'])
        checks.append({'sample_id':idx,'passed':not error,'issues':'; '.join(error),
                       'failed_xai_refreshes':xai_failed,'max_linf_255':attack['saved_linf_255']})
    report={'requested':requested,'completed':len(rows),'all_completed':len(rows)==requested,
            'all_technical_checks_pass':len(rows)==requested and all(c['passed'] for c in checks),
            'checks':checks,'human_asr_status':'separate review required'}
    atomic_json(Path(run_dir)/'pilot_validation.json',report)
    from IPython.display import display
    display(pd.DataFrame(checks))
    print('Pilot technical pass:',report['all_technical_checks_pass'])
    print('Human ASR: requires annotations in human_review_*.csv')
    return report

if RUN_DIR is not None:
    PILOT_VALIDATION=validate_pilot_results(RUN_DIR,MANIFEST_ROWS,len(PAIRS))


## Visualization — run on first completed pair (no model rerun)

In [ ]:
if RUN_DIR is not None and MANIFEST_ROWS:
    idx=MANIFEST_ROWS[0]['index']
    visualize_steps(RUN_DIR,idx)
    visualize_source_target_attention(RUN_DIR,idx)
    visualize_caption_space(RUN_DIR,idx)
    visualize_attention_timeline(RUN_DIR,idx)
    visualize_xai_dispersion(RUN_DIR,idx)
    visualize_loss_similarity(RUN_DIR,idx)
    visualize_concept_maps(RUN_DIR,idx,step=0)
    if CAPTION_EVALUATION:
        from IPython.display import display
        display(pd.DataFrame(CAPTION_EVALUATION))
    print('Output:',RUN_DIR)
